# Module 14 · Trajectory

Slingshot pseudotime on the transcriptional manifold, scores overlaid afterwards.

**The circularity constraint shapes every choice here.** If the trajectory is
fitted on a senescence-informed embedding, or rooted at the cell scoring highest
for senescence, then "senescence rises along pseudotime" restates the fit rather
than finding anything. So:

- the fit uses **`harmony` dims 1:30 and `mg_cluster` only** — no score enters;
- the root is chosen **marker-free**, by transcriptional entropy plus
  depth-corrected gene diversity, ranked across clusters. Not by a homeostatic
  panel score, which would be a biological quantity;
- scores are overlaid **post hoc**, for visualization and correlation only.

**Raw scores, not composite axes.** A composite of the form
`Score_state − Score_Homeostatic` shares its `−Homeostatic` term across states.
The root sits at the homeostatic end, so that shared term is itself a function
of pseudotime — a composite would correlate with pseudotime partly through it.
Module 10 section 15 tests this directly.

| Section | |
|---|---|
| 01-02 | config, inputs and cluster structure |
| 03 | marker-free root ranking |
| 04-05 | Slingshot fit, branch audit |
| 06-07 | branching tree, publication trajectory figure |
| 08 | trajectory × scores — post-hoc overlay |
| 09 | trajectory × state label |
| 10 | cell density along pseudotime, by group |
| 11-13 | progression panels |

> **Supersedes an earlier fit.** A previous version fitted on `seurat_clusters`
> at resolution 0.2 with the root set to the most homeostatic cluster by panel
> score. That root is score-informed. This version's entropy-based root is not,
> and is the one to use.

---
## 01 · Config

**Why.** Self-contained — the notebook carries its own config rather than
importing one. Paths come from the environment; see `.env.example`.

**`AXIS` is the one thing you change.** Column names, quadrant labels, contrast
names, figure titles and output directories derive from it, and outputs are
namespaced by axis so runs never overwrite each other.

```
AXIS <- "IRM"    # "IRM" | "DAM_like" | "ARM" | "Stress"
```

`STATE_ORDER` and `STATE_COLORS` name all five states regardless of `AXIS` —
that is the annotation registry, not a per-run choice.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
# Paths come from the environment - see .env.example.
#   SENESCENCE_DATA : analysis root
#   SENESCENCE_REF  : reference root (published panels, read-only)
# =============================================================================
SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")
if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")

# =============================================================================
# MODULE 05 — Senescence Enrichment & Cell Cycle Analysis
# CONFIG (Cell §0)
# =============================================================================
# Tests whether SnC cells show cell-cycle arrest and senescence pathway
# enrichment relative to Non-SnC. Reads M04's tissue-wide preprocessed .qs,
# subsets to one CELL_TYPE per run, scores cell cycle (Tirosh) + 10 module
# scores (Sloan Table S9), and runs five complementary models: Wilcoxon,
# RLM, OLS, LMM, and balanced OLS bootstrap.
#
# All run-time decisions live here. All cross-cell helpers are defined here
# so §1–§8 can use them without dependency-ordering issues.
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat("§0 — MODULE 05 CONFIG\n")
cat("=", strrep("=", 71), "\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────
options(repr.plot.width = 10, repr.plot.height = 5)


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")
CONDITION_TAG     <- if (IS_AGING) STUDY_TYPE else paste0(STUDY_TYPE, "_", DISEASE)
CONDITION_SUBPATH <- if (IS_AGING) STUDY_TYPE else file.path(STUDY_TYPE, DISEASE)


# ─────────────────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────────────────
SCRATCH <- SCRATCH

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = file.path(REF_DIR, "markers"),
    sloan_xlsx     = file.path(REF_DIR, "markers", "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(REF_DIR, "markers", "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(REF_DIR, "markers", "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Sloan Table S9 column mapping — 10 senescence gene lists
# Order is FIXED (used as canonical row order in §5 forest plots):
#   rows 1-8: individual hallmarks
#   rows 9-10: multi-hallmark composites
# ─────────────────────────────────────────────────────────────────────────────
SLOAN_HALLMARK_NAMES <- c(
    "p53_Targets",
    "CellCycleArrest",
    "SASP",
    "AntiApoptosis",
    "DDR",
    "CellSurfaceMarkers",
    "LysosomalContent",
    "SD_TMC",
    "SenMayo",
    "Fridman_Up"
)

SLOAN_LIST_TYPES <- c(
    rep("Individual hallmark", 7),
    rep("Multi-hallmark", 3)
)

# Color labels for module rows in §5 forests:
# hallmarks = dark gray, multi-hallmark composites = muted purple
SLOAN_HALLMARK_COLORS <- c(
    rep("#222222", 8),
    rep("#7B5BA3", 2)
)
names(SLOAN_HALLMARK_COLORS) <- SLOAN_HALLMARK_NAMES


# ─────────────────────────────────────────────────────────────────────────────
# Proliferation markers (M09 §6 → M05 §7)
# ─────────────────────────────────────────────────────────────────────────────
PROLIFERATION_MARKERS <- c(  
  # Core proliferation / mitotic markers
  "MKI67", "TOP2A", "HMGB2", "CENPF",
  "BIRC5", "CCNB1", "CCNB2", "UBE2C",

  # Canonical CDK inhibitors
  "CDKN2A",  # p16INK4A
  "CDKN1A",  # p21CIP1/WAF1
  "CDKN1B",  # p27KIP1
  "CDKN2B",  # p15INK4B

  # p53 pathway / DNA damage response
  "TP53",
  "GADD45A", "GADD45B", "GADD45G"
)


# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# EFFECT_CONFIGS — display rules for forest plots, dispatched by beta_scale
#
# Each model cell selects one of these configs to drive the inline forest
# plot. No hardcoding of axis units, label formatting, or null reference
# value across §4.x or §5.x cells.
#
#   probability_pts:  paired-difference β on the proportion scale (used by
#                     §4.x cell-cycle phase analyses).
#   score_units:      continuous module-score β (used by §5.x Sloan module
#                     score analyses, including lmer Gaussian).
#   log_odds:         log-odds β (binomial GLMM, used by §4.4 only).
#                     - log-scale x-axis showing OR
#                     - effect column shows "OR = 1.38"
#                     - null line at OR = 1
# ─────────────────────────────────────────────────────────────────────────────
EFFECT_CONFIGS <- list(
    probability_pts = list(
        scale          = "linear",
        null_value     = 0,
        x_label        = "beta (paired difference, SnC - Non-SnC)",
        eff_h_label    = "beta",
        ci_h_label     = "95% CI",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%+.3f", v)
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%+.3f, %+.3f]", lo, hi)
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) v
    ),
    score_units = list(
        scale          = "linear",
        null_value     = 0,
        x_label        = "beta (mean module score, SnC - Non-SnC)",
        eff_h_label    = "beta",
        ci_h_label     = "95% CI",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%+.3f", v)
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%+.3f, %+.3f]", lo, hi)
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) v
    ),
    log_odds = list(
        scale          = "log",
        null_value     = 1,
        x_label        = "Odds Ratio (SnC vs Non-SnC)",
        eff_h_label    = "OR",
        ci_h_label     = "95% CI (OR)",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%.2f", exp(v))
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%.2f, %.2f]", exp(lo), exp(hi))
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) exp(v)
    )
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}


# =============================================================================
# CROSS-CELL HELPERS
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)


# ─────────────────────────────────────────────────────────────────────────────
# Summary banner
# ─────────────────────────────────────────────────────────────────────────────
cat("\n  Run parameters:\n")
cat(sprintf("    TISSUE         : %s\n", TISSUE))
cat(sprintf("    STUDY_TYPE     : %s\n", STUDY_TYPE))
if (IS_DISEASE)
    cat(sprintf("    DISEASE        : %s\n", DISEASE))
cat(sprintf("    DATASET        : %s\n", DATASET))
cat(sprintf("    CELL_TYPE      : %s\n", CELL_TYPE))
cat(sprintf("    CONDITION_TAG  : %s\n", CONDITION_TAG))

cat(sprintf("\n  Inline plot viewport: %dx%d\n", 10, 5))

cat("\n  Stratification:\n")
cat(sprintf("    STRATIFY_BY_GROUP     : %s\n", STRATIFY_BY_GROUP))
if (STRATIFY_BY_GROUP) {
    cat(sprintf("    STRATIFICATION_GROUPS : %s\n",
                paste(STRATIFICATION_GROUPS, collapse = ", ")))
    cat(sprintf("    Total runs per cell   : 1 (\"All\") + %d strata = %d\n",
                length(STRATIFICATION_GROUPS), 1 + length(STRATIFICATION_GROUPS)))
} else {
    cat("    Stratified runs       : disabled (overall only)\n")
}

cat("\n  Statistical params:\n")
cat(sprintf("    min_cells_per_group : %d\n", STATISTICAL_PARAMS$min_cells_per_group))
cat(sprintf("    fdr_threshold       : %.2f (%s)\n",
            STATISTICAL_PARAMS$fdr_threshold, STATISTICAL_PARAMS$fdr_method))
cat(sprintf("    bootstrap_n_iter    : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter, STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("    confidence_level    : %.2f\n", STATISTICAL_PARAMS$confidence_level))

cat("\n  Effect display configs:\n")
for (nm in names(EFFECT_CONFIGS)) {
    cfg <- EFFECT_CONFIGS[[nm]]
    cat(sprintf("    %-15s -> axis=%s, null=%g, label='%s'\n",
                nm, cfg$scale, cfg$null_value, cfg$x_label))
}

cat("\n  Cross-cell helpers (defined in §0):\n")
cat("    Formatting   : fmt_n, fmt_size, fmt_pct, fmt_elapsed, fmt_p, fmt_p_short, sig_stars\n")
cat("    Time/save    : now_iso, bytes_str, time_step, save_figure, save_table\n")
cat("    Color        : text_color_for_bg\n")
cat("    Stratum      : filter_to_stratum\n")
cat("    Results      : tidy_model_results\n")
cat("    (Donor data helpers: build_donor_arms, build_donor_meta, tidy_lmm_term -- defined in §3.6)\n")

cat("\n  Senescence gene lists:\n")
cat(sprintf("    Source              : Sloan Table S9 (sheet 'Sen Gene Lists')\n"))
cat(sprintf("    Lists               : %d (8 hallmarks + 2 multi-hallmark)\n",
            length(SLOAN_HALLMARK_NAMES)))
cat(sprintf("    Fixed row order     : %s\n",
            paste(SLOAN_HALLMARK_NAMES, collapse = ", ")))

cat("\n  Proliferation markers (§7):\n")
cat(sprintf("    %s\n", paste(PROLIFERATION_MARKERS, collapse = ", ")))

cat("\n  Library versions:\n")
for (pkg in c("R", "Seurat", "lme4", "robustbase", "broom.mixed",
              "qs", "jsonlite")) {
    cat(sprintf("    %-12s : %s\n", pkg, R_PKG_VERSIONS[[pkg]]))
}

cat("\n  Paths:\n")
cat(sprintf("    M04 input root     : %s\n", PATHS$m04_root))
cat(sprintf("    M05 output root    : %s\n", PATHS$output_root))

cat("\n  Required input files:\n")
for (key in c("m04_seurat", "m04_manifest", "sloan_xlsx",
              "senmayo_xlsx", "fridman_gmt")) {
    exists_flag <- if (file.exists(PATHS[[key]])) "✓" else "✗"
    sz <- if (file.exists(PATHS[[key]])) fmt_size(PATHS[[key]]) else "MISSING"
    cat(sprintf("    %s  %-15s : %s  (%s)\n",
                exists_flag, key, PATHS[[key]], sz))
}

cat("\n=", strrep("=", 71), "\n", sep = "")
cat("✓ §0 config loaded. All cross-cell helpers in scope.\n")
cat("  Inline viewport: 10x5 (set via options(repr.plot.*)).\n")
cat("  Run §1 to load M04 manifest + .qs and validate.\n")

# =============================================================================
# §0.2 — AXIS SELECT
# =============================================================================
# The ONE thing you change to re-run the whole flow on a different state.
# Everything downstream — column names, quadrant labels, figure titles,
# legends, output paths — derives from this. Nothing is hardcoded per state.
# =============================================================================

AXIS <- "IRM"          # <<< "IRM" | "DAM_like" | "ARM" | "Stress"

# --- registry: score column candidates + display label + canonical hex -------
# Score columns are looked up in order; the first present on the object wins.
AXIS_REGISTRY <- list(
    # tag = short token used in CONTRAST NAMES and therefore in GSEA/DE filenames.
    # It is deliberately NOT the same as `lab` (display) or the list key: existing
    # results on disk are named DAMaxis_*, not DAM_likeaxis_*.
    IRM      = list(cols = c("Score_IRM",      "IRM1"),      lab = "IRM",      tag = "IRM",    hex = "#2980B9"),
    DAM_like = list(cols = c("Score_DAM_like", "DAM_like1"), lab = "DAM-like", tag = "DAM",    hex = "#C0392B"),
    ARM      = list(cols = c("Score_ARM",      "ARM1"),      lab = "ARM",      tag = "ARM",    hex = "#E67E22"),
    Stress   = list(cols = c("Score_Stress",   "Stress1"),   lab = "Stress",   tag = "Stress", hex = "#8E44AD")
)
stopifnot(AXIS %in% names(AXIS_REGISTRY))

AXIS_SPEC <- AXIS_REGISTRY[[AXIS]]
X_LAB     <- AXIS_SPEC$lab
X_HEX     <- AXIS_SPEC$hex
Y_LAB     <- "Senescence"
Y_TAG     <- "SnC"
X_TAG     <- AXIS_SPEC$tag
SEN_CANDIDATES <- c("senescence_score", "SenePy_score")

# --- quadrant labels derive from the axis -----------------------------------
QUAD_COL    <- paste0("quad4_", tolower(AXIS))
QUAD_LEVELS <- c(sprintf("Sen- %s-", X_LAB), sprintf("Sen+ %s-", X_LAB),
                 sprintf("Sen- %s+", X_LAB), sprintf("Sen+ %s+", X_LAB))
QUAD_COLORS <- setNames(c("#B8B8B8", "#2E7D32", X_HEX, "#6A1B9A"), QUAD_LEVELS)

# --- the four DE / GSEA contrasts, named off the tags ------------------------
CONTRASTS <- c(sprintf("%saxis_%spos", X_TAG, Y_TAG),   # axis effect within SnC+
               sprintf("%saxis_%sneg", X_TAG, Y_TAG),   # axis effect within SnC-
               sprintf("%saxis_%spos", Y_TAG, X_TAG),   # sen effect within axis+
               sprintf("%saxis_%sneg", Y_TAG, X_TAG))   # sen effect within axis-
CONTRAST_HEADERS <- c(
    sprintf("%s+%s+ vs %s+%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s−%s+ vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s+ vs %s−%s+", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s− vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB))

# --- outputs are namespaced by axis so runs never overwrite each other -------
AXIS_FIG_DIR <- file.path(PATHS$figures, AXIS)
AXIS_RES_DIR <- file.path(PATHS$results, AXIS)
dir.create(AXIS_FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(AXIS_RES_DIR, recursive = TRUE, showWarnings = FALSE)

# --- state annotation (all five; NOT gated by AXIS) --------------------------
STATE_ORDER  <- c("Homeostatic", "ARM", "IRM", "Stress", "DAM_like")
STATE_COLORS <- c(Homeostatic = "#7F8C8D", ARM = "#E67E22", IRM = "#2980B9",
                  Stress = "#8E44AD", DAM_like = "#C0392B")

# --- DE model design (confirmed 2026-07-28; supersedes the leaner ~pop+grp2+Sex)
DE_DESIGN     <- "~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort"
DE_COEF       <- "popTEST"
DE_MIN_CELLS  <- 10L

# --- GSEA (consumed by the python notebook via gsea_config.json) -------------
GSEA_DBS      <- c("Reactome_2022")
GSEA_FDR_SIG  <- 0.05
GSEA_N_COMMON <- 6L      # sig in >=3 contrasts, top N by mean NES
GSEA_N_UNIQUE <- 4L      # sig in exactly 1 contrast, top N by |NES|
GSEA_RIBO_STRIP <- FALSE # keep translational terms; see Part F Why



# §0.3 — AXIS-AWARE HELPERS  (one definition each — see note)
# =============================================================================
# In the source notebook fit_one was defined 13x, resolve_score_col 6x,
# z_score 5x, venn2 4x, and save_figure was REDEFINED at cells 317/341,
# shadowing the canonical version above. Everything lives here now so a
# later cell cannot silently shadow it.
# =============================================================================

# --- resolve a score column from candidates ---------------------------------
resolve_score_col <- function(md, candidates, what = "score") {
    hit <- candidates[candidates %in% colnames(md)]
    if (!length(hit)) stop(sprintf("no %s column found; tried: %s",
                                   what, paste(candidates, collapse = ", ")))
    hit[1]
}

z_score <- function(x) as.numeric(scale(x))

# --- build the SnC x AXIS quadrant column -----------------------------------
# Mean-split on z-scored values, exactly as the source (cell 54).
build_quadrants <- function(obj, axis = AXIS, verbose = TRUE) {
    md   <- obj@meta.data
    spec <- AXIS_REGISTRY[[axis]]
    sen  <- resolve_score_col(md, SEN_CANDIDATES, "senescence")
    xcol <- resolve_score_col(md, spec$cols, paste(axis, "score"))
    sz <- z_score(md[[sen]]); xz <- z_score(md[[xcol]])
    lab <- spec$lab
    q <- ifelse(sz >  0 & xz >  0, sprintf("Sen+ %s+", lab),
        ifelse(sz >  0 & xz <= 0, sprintf("Sen+ %s-", lab),
        ifelse(sz <= 0 & xz >  0, sprintf("Sen- %s+", lab),
                                  sprintf("Sen- %s-", lab))))
    q[is.na(sz) | is.na(xz)] <- NA
    obj[[paste0("quad4_", tolower(axis))]] <- q
    obj$sen_z <- sz
    obj$axis_z <- xz
    if (verbose) {
        cat(sprintf("  scores : sen=%s  %s=%s\n", sen, axis, xcol))
        print(table(q, useNA = "ifany"))
        cat(sprintf("  cor(sen_z, %s_z) = %.3f   <- independence check\n",
                    tolower(axis), cor(sz, xz, use = "complete.obs")))
    }
    obj
}

# --- PREFLIGHT: fail loudly before any model runs ---------------------------
preflight_axis <- function(obj, axis = AXIS, min_cells = 50L, donor_col = "Donor") {
    md <- obj@meta.data; ok <- TRUE
    say <- function(pass, msg) {
        cat(sprintf("  [%s] %s\n", if (pass) "OK  " else "FAIL", msg))
        if (!pass) ok <<- FALSE
    }
    cat(sprintf("\n── PREFLIGHT · axis = %s ──\n", axis))
    say(axis %in% names(AXIS_REGISTRY), sprintf("axis '%s' is registered", axis))
    spec <- AXIS_REGISTRY[[axis]]
    xhit <- spec$cols[spec$cols %in% colnames(md)]
    say(length(xhit) > 0, sprintf("score column present (%s)",
        if (length(xhit)) xhit[1] else paste(spec$cols, collapse = "/")))
    shit <- SEN_CANDIDATES[SEN_CANDIDATES %in% colnames(md)]
    say(length(shit) > 0, "senescence score column present")
    if (length(xhit) && length(shit)) {
        say(sd(md[[xhit[1]]], na.rm = TRUE) > 0, "axis score is non-constant")
        say(sd(md[[shit[1]]], na.rm = TRUE) > 0, "senescence score is non-constant")
    }
    qc <- paste0("quad4_", tolower(axis))
    if (qc %in% colnames(md)) {
        tb <- table(md[[qc]])
        say(length(tb) == 4, sprintf("all four quadrants populated (%d)", length(tb)))
        say(all(tb >= min_cells), sprintf("every quadrant >= %d cells (min %d)",
                                          min_cells, min(tb)))
        if (donor_col %in% colnames(md)) {
            nd <- tapply(md[[donor_col]], md[[qc]], function(z) length(unique(z)))
            say(all(nd >= 2), sprintf("every quadrant has >=2 donors (min %d)", min(nd)))
        }
    } else cat(sprintf("  [--  ] %s not built yet (run B1)\n", qc))
    cat(sprintf("── %s ──\n\n", if (ok) "PASS" else "STOP: fix before proceeding"))
    invisible(ok)
}

# --- ONE mixed-model fitter (replaces 13 copies of fit_one) -----------------
# formula_str is built by the caller, so every Part C analysis is this
# function with a different formula and a different subset.
fit_lmm <- function(df, formula_str, term, label = NA_character_) {
    fit <- tryCatch(lmerTest::lmer(as.formula(formula_str), data = df,
                                   REML = TRUE,
                                   control = lme4::lmerControl(
                                       optimizer = "bobyqa",
                                       optCtrl = list(maxfun = 2e5))),
                    error = function(e) NULL, warning = function(w) NULL)
    if (is.null(fit)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    co <- summary(fit)$coefficients
    if (!term %in% rownames(co)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    b <- co[term, "Estimate"]; s <- co[term, "Std. Error"]
    data.frame(label = label, term = term, beta = b, se = s,
               ci_low = b - 1.96 * s, ci_high = b + 1.96 * s,
               p_value = co[term, "Pr(>|t|)"], n = nrow(df), converged = TRUE)
}

# --- ONE forest renderer (replaces 7 near-copies) ---------------------------
forest_plot <- function(res, title = "", xlab = "beta (95% CI)",
                        facet = NULL, color = X_HEX) {
    stopifnot(all(c("label", "beta", "ci_low", "ci_high") %in% names(res)))
    if (!"p_adj" %in% names(res))
        res$p_adj <- p.adjust(res$p_value, method = STATISTICAL_PARAMS$fdr_method)
    res$sig <- sig_stars(res$p_adj)
    res$label <- factor(res$label, levels = rev(unique(res$label)))
    p <- ggplot(res, aes(x = beta, y = label)) +
        geom_vline(xintercept = 0, linetype = "dashed",
                   colour = "grey60", linewidth = 0.3) +
        geom_errorbarh(aes(xmin = ci_low, xmax = ci_high),
                       height = 0, linewidth = 0.4, colour = color) +
        geom_point(size = 1.8, colour = color) +
        geom_text(aes(x = ci_high, label = sig), hjust = -0.35,
                  size = 2.6, na.rm = TRUE) +
        labs(title = title, x = xlab, y = NULL) +
        theme_clean() +
        theme(panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.25))
    if (!is.null(facet)) p <- p + facet_wrap(as.formula(paste("~", facet)), scales = "free_x")
    p + coord_cartesian(clip = "off")
}

# --- block banner: prints the Why with the axis resolved --------------------
say_block <- function(id, title, why = NULL) {
    cat("\n", strrep("═", 76), "\n", sep = "")
    cat(sprintf("%s  ·  %s\n", id, sprintf(title, X_LAB)))
    cat(strrep("═", 76), "\n", sep = "")
    if (!is.null(why)) cat(sprintf("WHY: %s\n\n", sprintf(why, X_LAB)))
}

# X_COL is resolved once the object exists, in the load section:
#     X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis')
# Every downstream cell reads X_COL, never a literal score column.

---
## 02 · Inputs and cluster structure

**Why.** States the fit space, the display space and the cluster labels, then reports cluster sizes and sequencing depth — **technical QC only, no biological scores**. That the diagnostic itself is score-free is part of the guarantee.

**Fit space.** `harmony` dims 1:30. **Display.** `umap.mg`. **Clusters.** `mg_cluster`.

In [ ]:
mg

In [ ]:
# same cells going in?
cat("n cells:", ncol(mg), "\n")
cat("barcode checksum:", digest::digest(sort(colnames(mg))), "\n")
# is the harmony embedding the thing that moved?
cat("harmony checksum:", digest::digest(round(Embeddings(mg,"harmony")[,1:5], 4)), "\n")
cat("mg_cluster table:\n"); print(table(mg$mg_cluster))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PSEUDOTIME STEP 1 — inputs only. NO senescence/state scores touch the fit.
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

CLUSTER_LBL <- "mg_cluster"
LIN_RED     <- "harmony"       # fit space
DISP_RED    <- "umap.mg"       # display space
N_DIMS      <- 30

cat("cells:", ncol(mg), "| reductions:", paste(Reductions(mg), collapse=", "), "\n")
stopifnot(LIN_RED %in% Reductions(mg), DISP_RED %in% Reductions(mg))
stopifnot(CLUSTER_LBL %in% colnames(mg@meta.data))
cat("fit space :", LIN_RED, "dims 1:", N_DIMS, "of", ncol(Embeddings(mg, LIN_RED)), "\n")
cat("display   :", DISP_RED, "\n")
cat("NA clusters:", sum(is.na(mg@meta.data[[CLUSTER_LBL]])), "\n\n")

# cluster sizes + technical QC only (no biological scores)
md <- mg@meta.data
str_tab <- data.frame(clus = factor(md[[CLUSTER_LBL]]),
                      nCount = md$nCount_RNA, nFeature = md$nFeature_RNA) %>%
    group_by(clus) %>%
    summarise(n = n(),
              med_UMI  = round(median(nCount)),
              med_gene = round(median(nFeature)), .groups="drop") %>%
    arrange(desc(n))
cat("════ cluster structure (size + depth only) ════\n")
print(as.data.frame(str_tab), row.names=FALSE)
cat(sprintf("\n%d clusters | smallest n=%d (cl.%s)\n",
    nrow(str_tab), min(str_tab$n), str_tab$clus[which.min(str_tab$n)]))

---
## 03 · Marker-free root ranking

**Why.** The root has to come from somewhere, and any biological marker used to pick it contaminates every correlation computed against the resulting pseudotime. So the root is chosen from two properties of the transcriptome that carry no state information:

**Transcriptional entropy** — `−Σ p log p` over the per-cell normalized count distribution. High entropy means expression spread across many genes, which is the usual signature of a less differentiated state.

**Depth-corrected gene diversity** — residuals of `n_genes ~ log1p(total_counts)`, so a cell is not called diverse merely for being deeply sequenced.

Clusters are ranked on the mean of the two ranks; the top cluster is the root candidate.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PSEUDOTIME STEP 2 — root by unbiased stemness (marker-free)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Matrix); library(dplyr) })
cts  <- GetAssayData(mg, layer="counts")
clus <- as.character(mg@meta.data[[CLUSTER_LBL]])
tot  <- Matrix::colSums(cts)

n_genes     <- Matrix::colSums(cts > 0)
resid_genes <- residuals(lm(n_genes ~ log1p(tot)))     # depth-corrected diversity
p    <- cts %*% Diagonal(x = 1/tot)
logp <- p; logp@x <- log(logp@x)
entropy <- -Matrix::colSums(p * logp)                  # transcriptional entropy

rt <- data.frame(clus=clus, entropy=entropy, resid_genes=resid_genes) %>%
    group_by(clus) %>%
    summarise(n=n(), entropy=round(mean(entropy),3),
              resid_genes=round(mean(resid_genes),1), .groups="drop") %>%
    mutate(root_rank = (rank(-entropy) + rank(-resid_genes))/2) %>%
    arrange(root_rank)
cat("════ unbiased root ranking (top = least differentiated) ════\n")
print(as.data.frame(rt), row.names=FALSE)
cat(sprintf("\n→ root candidate: cl.%s\n", rt$clus[1]))

---
## 04 · Slingshot fit

**Why.** Slingshot fits a minimum spanning tree over cluster centroids, then principal curves through the cells. Fitting on centroids is what makes the topology reproducible.

**Inputs.** `harmony` dims 1:30 and `mg_cluster`. Nothing else — no score, no group label, no senescence call.

**Seeding note.** The clustering is unseeded but reproducible when run in order from a fresh kernel. Adding `set.seed()` changes the RNG state and yields a different clustering — it replaces the result rather than stabilizing it.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PSEUDOTIME STEP 3 — Slingshot fit. Harmony + mg_cluster only. No scores.
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; N_DIMS <- 30
ROOT_CLUS   <- "8"                       # homeostatic root (stated in methods)

sce <- as.SingleCellExperiment(mg)
RD  <- grep("harmony", reducedDimNames(sce), ignore.case=TRUE, value=TRUE)[1]
stopifnot(!is.na(RD), ROOT_CLUS %in% as.character(colData(sce)[[CLUSTER_LBL]]))
cat("fit:", RD, "dims 1:", N_DIMS, "| clusters:", CLUSTER_LBL, "| root: cl.", ROOT_CLUS, "\n\n")

sce <- slingshot(sce,
                 clusterLabels = colData(sce)[[CLUSTER_LBL]],
                 reducedDim    = reducedDim(sce, RD)[, 1:N_DIMS],
                 start.clus    = ROOT_CLUS)

sds  <- SlingshotDataSet(sce); lins <- slingLineages(sds); pt <- slingPseudotime(sds)
cat(sprintf("%d lineages from cl.%s:\n", length(lins), ROOT_CLUS))
for (i in seq_along(lins)) cat(sprintf("  L%d: %s\n", i, paste(lins[[i]], collapse="\u2192")))
cat(sprintf("\npseudotime range: %.1f – %.1f | cells assigned: %d / %d\n",
    min(pt, na.rm=TRUE), max(pt, na.rm=TRUE),
    sum(rowSums(!is.na(pt)) > 0), nrow(pt)))

---
## 05 · Branch audit

**Why.** **Structure only** — trunk, split points, monotonicity, lineage sizes. Slingshot returns every lineage the MST supports, including ones that double back or contain a handful of cells. Auditing before overlaying anything is what stops a spurious branch carrying a correlation into section 08.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BRANCH AUDIT — structure only (trunk, split points, monotonicity, sizes)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(slingshot); library(SingleCellExperiment) })
CLUSTER_LBL <- "mg_cluster"

sds  <- SlingshotDataSet(sce); lins <- slingLineages(sds); pt <- slingPseudotime(sds)
clus <- as.character(colData(sce)[[CLUSTER_LBL]])
paths <- lapply(lins, as.character)

# shared trunk = longest common prefix
trunk <- Reduce(function(a,b){ k<-0
  for (j in seq_len(min(length(a),length(b)))) if (a[j]==b[j]) k<-j else break
  a[seq_len(k)] }, paths)
cat("trunk:", paste(trunk, collapse="\u2192"), "\n\n")

# branch points: a cluster leading to >1 distinct successor
cat("════ branch points ════\n")
allc <- unique(unlist(paths))
for (cl in allc) {
  nxt <- unique(unlist(lapply(paths, function(p){ k<-which(p==cl)
    if (length(k) && k<length(p)) p[k+1] else NULL })))
  if (length(nxt) > 1)
    cat(sprintf("  cl.%-3s (n=%5d) \u2192 {%s}\n", cl, sum(clus==cl), paste(nxt, collapse=", ")))
}
leaves <- unique(sapply(paths, function(p) p[length(p)]))
cat("\nterminals:", paste(sprintf("cl.%s(n=%d)", leaves, sapply(leaves, function(x) sum(clus==x))),
                          collapse=", "), "\n\n")

# monotonicity per lineage
cat("════ lineage validity ════\n")
val <- do.call(rbind, lapply(seq_along(lins), function(i){
  p <- paths[[i]]
  mu <- sapply(p, function(cl) mean(pt[clus==cl, i], na.rm=TRUE))
  ns <- sapply(p, function(cl) sum(clus==cl & is.finite(pt[,i])))
  dr <- if (length(mu)>1) diff(mu) else 0
  data.frame(L=i, path=paste(p, collapse="\u2192"),
             pt_start=round(mu[1],1), pt_end=round(mu[length(mu)],1),
             max_drop=round(min(dr),1), monotonic=min(dr) >= -2.5,
             min_n=min(ns))
}))
print(val, row.names=FALSE)
cat(sprintf("\nmonotonic: %d/%d | lineages with a cluster <300 cells: %s\n",
    sum(val$monotonic), nrow(val),
    paste0("L", val$L[val$min_n < 300], collapse=",")))

---
## 06 · Branching tree

**Why.** The topology as an abstract dendrogram alongside the matching UMAP, so the tree and the embedding can be checked against each other.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BRANCHING TREE — abstract dendrogram (readable) + matching UMAP
#   x = mean pseudotime · shared trunk grey · each lineage its own colour
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; DISP_RED <- "umap.mg"
VALID <- c(1,2,3,5,6,7)                      # L4 excluded

sds   <- SlingshotDataSet(sce)
paths <- lapply(slingLineages(sds), as.character)
pt    <- slingPseudotime(sds)
clus  <- as.character(mg@meta.data[[CLUSTER_LBL]])
ptc   <- rowMeans(pt, na.rm = TRUE)

# ── edges + which lineages use each ─────────────────────────────────────────
E <- unique(do.call(rbind, lapply(paths[VALID], function(p)
      data.frame(from = p[-length(p)], to = p[-1], stringsAsFactors = FALSE))))
E$n_lin <- sapply(seq_len(nrow(E)), function(r)
  sum(sapply(paths[VALID], function(p)
    any(p[-length(p)] == E$from[r] & p[-1] == E$to[r]))))
E$lin <- sapply(seq_len(nrow(E)), function(r){
  w <- which(sapply(paths[VALID], function(p)
    any(p[-length(p)] == E$from[r] & p[-1] == E$to[r])))
  if (length(w) == 1) VALID[w] else NA })

LCOL <- setNames(c("#C0392B","#E67E22","#2980B9","#16A085","#8E44AD","#7F8C8D"), VALID)
ecol <- ifelse(is.na(E$lin), "grey35", LCOL[as.character(E$lin)])
elwd <- ifelse(is.na(E$lin), 4.5, 2.6)

# ── layout: DFS terminal order → y; mean pseudotime → x ─────────────────────
kids  <- split(E$to, E$from)
nodes <- unique(c(E$from, E$to))
root  <- setdiff(E$from, E$to)[1]
dfs   <- function(nd) { ch <- kids[[nd]]
          if (is.null(ch)) nd else unlist(lapply(ch, dfs)) }
terms <- dfs(root)
yt    <- setNames(seq_along(terms), terms)
ny    <- function(nd) { ch <- kids[[nd]]
          if (is.null(ch)) yt[[nd]] else mean(sapply(ch, ny)) }
Y <- setNames(sapply(nodes, ny), nodes)
X <- setNames(sapply(nodes, function(cl) mean(ptc[clus == cl], na.rm = TRUE)), nodes)
N <- setNames(sapply(nodes, function(cl) sum(clus == cl)), nodes)

options(repr.plot.width = 13, repr.plot.height = 6)
par(mfrow = c(1,2), mar = c(4,3,3,1))

# ── PANEL 1 · abstract tree ─────────────────────────────────────────────────
plot(NA, xlim = range(X)+c(-4,6), ylim = c(0.4, length(terms)+0.6),
     xlab = "mean pseudotime", ylab = "", yaxt = "n", bty = "n",
     main = "Branching structure", cex.main = 1)
for (r in seq_len(nrow(E))) {            # elbow connectors — never cross
  f <- E$from[r]; t <- E$to[r]
  segments(X[f], Y[f], X[f], Y[t], col = ecol[r], lwd = elwd[r], lend = 1)
  segments(X[f], Y[t], X[t], Y[t], col = ecol[r], lwd = elwd[r], lend = 1)
}
points(X[nodes], Y[nodes], pch = 21, bg = "white", col = "black", cex = 3.1, lwd = 1.3)
text(X[nodes], Y[nodes], nodes, font = 2, cex = 0.78)
text(X[nodes], Y[nodes] + 0.30, sprintf("n=%d", N[nodes]), cex = 0.55, col = "grey40")
text(X[root], Y[root], "\u2605", col = "#1B7837", cex = 2.4)
for (t in terms) text(X[t] + 3.2, Y[t],
  sprintf("L%d", VALID[which(sapply(paths[VALID], function(p) tail(p,1) == t))[1]]),
  cex = 0.75, font = 2, col = LCOL[as.character(
    VALID[which(sapply(paths[VALID], function(p) tail(p,1) == t))[1]])])
legend("topleft", bty = "n", cex = 0.7, lwd = c(4.5, 2.6), col = c("grey35","#C0392B"),
       legend = c("shared trunk", "lineage-specific"))

# ── PANEL 2 · same tree on UMAP ─────────────────────────────────────────────
dimred <- Embeddings(mg, DISP_RED)[,1:2]
cent   <- t(sapply(sort(unique(clus)), function(i) colMeans(dimred[clus==i,,drop=FALSE])))
rownames(cent) <- sort(unique(clus))
plot(dimred, col = "grey88", pch = 16, cex = 0.28, asp = 1,
     xlab = "UMAP 1", ylab = "UMAP 2", main = "Same tree in UMAP space", cex.main = 1)
for (r in seq_len(nrow(E)))
  lines(cent[c(E$from[r], E$to[r]),], col = ecol[r], lwd = elwd[r])
points(cent[nodes,,drop=FALSE], pch = 21, bg = "white", col = "black", cex = 2.2, lwd = 1.2)
text(cent[nodes,1], cent[nodes,2], nodes, font = 2, cex = 0.68)
text(cent[root,1], cent[root,2], "\u2605", col = "#1B7837", cex = 2.6)

dev.copy(png, "mg_branching_tree_clean.png", width = 1560, height = 720, res = 120); dev.off()
cat(sprintf("root cl.%s | trunk %s | terminals %s\n", root,
    paste(names(which(E$n_lin == max(E$n_lin))), collapse=","), paste(terms, collapse=",")))

---
## 07 · Trajectory figure

**Why.** The publication panel — trajectory on `umap.mg` with lineage connectors.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MICROGLIAL TRAJECTORY — publication figure
#   nodes  : green = root · gold = branch point · red = terminal · white = transit
#   edges  : blue/purple/pink/brown per lineage · width ∝ cells downstream
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; DISP_RED <- "umap.mg"
VALID    <- c(1,2,3,5,6,7)
MIN_GAP  <- 1.5
NODE     <- 2.3
ROOT_COL <- "#1B7837"; BP_COL <- "#E8B31F"; TERM_COL <- "#C0392B"
EPAL     <- c("#0072B2","#6A3D9A","#CC79A7","#56B4E9","#8C564B","#3B7C8E")
TRUNK    <- "#555555"

sds   <- SlingshotDataSet(sce)
paths <- lapply(slingLineages(sds), as.character)[VALID]
ptc   <- rowMeans(slingPseudotime(sds), na.rm = TRUE)
clus  <- as.character(mg@meta.data[[CLUSTER_LBL]])

E <- unique(do.call(rbind, lapply(paths, function(p)
       data.frame(from = p[-length(p)], to = p[-1], stringsAsFactors = FALSE))))
uses <- lapply(seq_len(nrow(E)), function(r)
  which(vapply(paths, function(p)
    any(p[-length(p)] == E$from[r] & p[-1] == E$to[r]), logical(1))))
E$n_lin <- lengths(uses)
E$lin   <- vapply(uses, function(u) if (length(u)==1) u else NA_integer_, integer(1))

kids   <- split(E$to, E$from); parent <- setNames(E$from, E$to)
nodes  <- unique(c(E$from, E$to)); root <- setdiff(E$from, E$to)[1]
bpts   <- names(kids)[lengths(kids) > 1]
LCOL   <- setNames(EPAL[seq_along(VALID)], seq_along(VALID))
N      <- setNames(vapply(nodes, function(c_) sum(clus==c_), numeric(1)), nodes)

# ── flow: cells in the subtree below each edge → edge thickness ─────────────
subN <- function(nd){ ch <- kids[[nd]]
  if (is.null(ch)) N[[nd]] else N[[nd]] + sum(vapply(ch, subN, numeric(1))) }
E$flow <- vapply(E$to, subN, numeric(1))
elwd   <- 0.9 + 5.2 * sqrt(E$flow / max(E$flow))
ecol   <- ifelse(is.na(E$lin), TRUNK, LCOL[as.character(E$lin)])

# ── layout ──────────────────────────────────────────────────────────────────
dfs   <- function(nd) if (is.null(kids[[nd]])) nd else unlist(lapply(kids[[nd]], dfs))
terms <- dfs(root); yt <- setNames(seq_along(terms), terms)
Yl <- list(); setY <- function(nd){
  y <- if (is.null(kids[[nd]])) yt[[nd]] else mean(vapply(kids[[nd]], setY, numeric(1)))
  Yl[[nd]] <<- y; y }
invisible(setY(root)); Y <- unlist(Yl)[nodes]

Xr <- setNames(vapply(nodes, function(c_) mean(ptc[clus==c_], na.rm=TRUE), numeric(1)), nodes)
bfs <- root; i <- 1
while (i <= length(bfs)) { if (!is.null(kids[[bfs[i]]])) bfs <- c(bfs, kids[[bfs[i]]]); i <- i+1 }
X <- Xr; for (nd in bfs[-1]) X[nd] <- max(Xr[nd], X[parent[[nd]]] + MIN_GAP)

# lineage cell totals → identify the main lineage
lin_n <- vapply(paths, function(p) sum(N[p]), numeric(1))
main_i <- which.max(lin_n)

nbg <- setNames(rep("white", length(nodes)), nodes)
nbg[terms] <- TERM_COL; nbg[bpts] <- BP_COL; nbg[root] <- ROOT_COL
nfg <- ifelse(nbg %in% c(TERM_COL, ROOT_COL), "white", "grey10")

# ── x-axis: ticks bounded to the data, rounded to 5 ─────────────────────────
xd   <- range(X)
tk   <- seq(floor(xd[1]/5)*5, ceiling(xd[2]/5)*5, by = 5)
tk   <- tk[tk >= xd[1] - 1 & tk <= xd[2] + 1]
lab0 <- max(X) + 3.0; xlim <- c(min(X) - 1.5, lab0 + 22)
ybot <- 0.30

options(repr.plot.width = 13, repr.plot.height = 5.4)
layout(matrix(1:2, 1), widths = c(1.18, 1))

# ── LEFT · dendrogram ───────────────────────────────────────────────────────
par(mar = c(4.0, 0.8, 2.4, 0.4), xpd = NA)
plot(NA, xlim = xlim, ylim = c(ybot - 0.55, length(terms) + 0.9),
     xlab = "", ylab = "", axes = FALSE)
segments(min(tk), ybot, max(tk), ybot, col = "grey55", lwd = 0.8)      # axis line, data only
segments(tk, ybot, tk, ybot - 0.10, col = "grey55", lwd = 0.8)          # ticks
text(tk, ybot - 0.30, tk, cex = 0.66, col = "grey30")
text(mean(range(tk)), ybot - 0.55, "pseudotime (cluster mean)",
     cex = 0.72, col = "grey30")

for (r in seq_len(nrow(E))) {
  f <- E$from[r]; t <- E$to[r]
  segments(X[f], Y[f], X[f], Y[t], col = ecol[r], lwd = elwd[r], lend = 1)
  segments(X[f], Y[t], X[t], Y[t], col = ecol[r], lwd = elwd[r], lend = 1)
}
points(X, Y, pch = 21, bg = nbg[nodes], col = "grey20", cex = NODE, lwd = 1)
text(X, Y, nodes, font = 2, cex = 0.56, col = nfg[nodes])

for (k in seq_along(terms)) {
  t  <- terms[k]
  li <- which(vapply(paths, function(p) tail(p,1) == t, logical(1)))[1]
  segments(X[t] + 0.8, Y[t], lab0 - 0.4, Y[t], col = "grey88", lwd = 0.4, lty = 3)
  text(lab0, Y[t], sprintf("L%d%s", VALID[li], if (li == main_i) " \u25c0" else ""),
       adj = 0, font = 2, cex = 0.68, col = LCOL[as.character(li)])
  text(lab0 + 5.4,  Y[t], sprintf("cl.%s", t), adj = 0, cex = 0.62, col = "grey35")
  text(lab0 + 11.5, Y[t], format(N[t], big.mark = ","), adj = 1, cex = 0.6, col = "grey55")
}
text(lab0 + 11.5, length(terms) + 0.72, "cells", adj = 1, cex = 0.55, col = "grey60", font = 3)
mtext("Branching structure", 3, line = 0.55, adj = 0, cex = 0.92, font = 2)
legend(min(X) - 1.5, ybot - 0.75, horiz = TRUE, bty = "n", cex = 0.62, pt.cex = 1.35,
       pch = 21, pt.bg = c(ROOT_COL, BP_COL, TERM_COL, "white"), col = "grey20",
       legend = c("root", "branch point", "terminal", "transit"))

# ── RIGHT · same tree on UMAP ───────────────────────────────────────────────
dimred <- Embeddings(mg, DISP_RED)[, 1:2]
cent <- t(vapply(nodes, function(i) colMeans(dimred[clus==i, , drop=FALSE]), numeric(2)))
rownames(cent) <- nodes
par(mar = c(4.0, 2.6, 2.4, 0.4), xpd = FALSE)
plot(dimred, col = "grey91", pch = 16, cex = 0.2, asp = 1, axes = FALSE, xlab="", ylab="")
for (r in seq_len(nrow(E)))
  lines(cent[c(E$from[r], E$to[r]), ], col = ecol[r], lwd = elwd[r] * 0.85)
points(cent, pch = 21, bg = nbg[nodes], col = "grey20", cex = NODE*0.82, lwd = 1)
text(cent[,1], cent[,2], nodes, font = 2, cex = 0.5, col = nfg[nodes])
mtext("UMAP", 3, line = 0.55, adj = 0, cex = 0.92, font = 2)
mtext("UMAP 1", 1, line = 1.0, cex = 0.66, col = "grey45")
mtext("UMAP 2", 2, line = 0.6, cex = 0.66, col = "grey45")

dev.copy(png, "mg_trajectory_publication.png", width = 1950, height = 810, res = 150); dev.off()
dev.copy(svg, "mg_trajectory_publication.svg", width = 13, height = 5.4); dev.off()

cat(sprintf("main lineage: L%d (%s cells) \u2014 %s\n", VALID[main_i],
    format(lin_n[main_i], big.mark=","), paste0("cl.", paths[[main_i]], collapse="\u2192")))
cat("\ncells per lineage:\n")
for (i in seq_along(paths))
  cat(sprintf("  L%-2d %6s  %s\n", VALID[i], format(lin_n[i], big.mark=","),
              paste0("cl.", paths[[i]], collapse="\u2192")))
cat("\nbranch points: ")
cat(paste(sprintf("cl.%s\u2192{%s}", bpts, vapply(bpts, function(b)
    paste(kids[[b]], collapse=","), character(1))), collapse = "  "), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MICROGLIAL TRAJECTORY — publication figure
#   nodes : green = root · gold = branch · red = terminal · white = transit
#   edges : curved + haloed, width ∝ cells downstream, siblings fan apart
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; DISP_RED <- "umap.mg"
VALID    <- c(1,2,3,5,6,7)
MIN_GAP  <- 1.5;  NODE <- 2.3
UMAP_BG  <- "grey"          # "grey" or "pseudotime"
BOW      <- 0.20            # edge curvature (0 = straight)
ROOT_COL <- "#1B7837"; BP_COL <- "#E8B31F"; TERM_COL <- "#C0392B"
EPAL     <- c("#0072B2","#6A3D9A","#CC79A7","#56B4E9","#8C564B","#3B7C8E")
TRUNK    <- "#4D4D4D"

sds   <- SlingshotDataSet(sce)
paths <- lapply(slingLineages(sds), as.character)[VALID]
ptc   <- rowMeans(slingPseudotime(sds), na.rm = TRUE)
clus  <- as.character(mg@meta.data[[CLUSTER_LBL]])

E <- unique(do.call(rbind, lapply(paths, function(p)
       data.frame(from = p[-length(p)], to = p[-1], stringsAsFactors = FALSE))))
uses <- lapply(seq_len(nrow(E)), function(r)
  which(vapply(paths, function(p)
    any(p[-length(p)] == E$from[r] & p[-1] == E$to[r]), logical(1))))
E$lin <- vapply(uses, function(u) if (length(u)==1) u else NA_integer_, integer(1))

kids  <- split(E$to, E$from); parent <- setNames(E$from, E$to)
nodes <- unique(c(E$from, E$to)); root <- setdiff(E$from, E$to)[1]
bpts  <- names(kids)[lengths(kids) > 1]
LCOL  <- setNames(EPAL[seq_along(VALID)], seq_along(VALID))
N     <- setNames(vapply(nodes, function(c_) sum(clus==c_), numeric(1)), nodes)

subN  <- function(nd){ ch <- kids[[nd]]
  if (is.null(ch)) N[[nd]] else N[[nd]] + sum(vapply(ch, subN, numeric(1))) }
E$flow <- vapply(E$to, subN, numeric(1))
elwd   <- 0.9 + 5.0 * sqrt(E$flow / max(E$flow))
ecol   <- ifelse(is.na(E$lin), TRUNK, LCOL[as.character(E$lin)])

# ── siblings fan apart: bow scaled by child index within parent ─────────────
E$bow <- vapply(seq_len(nrow(E)), function(r){
  sib <- kids[[E$from[r]]]; k <- length(sib)
  if (k < 2) return(0)
  seq(-BOW, BOW, length.out = k)[match(E$to[r], sib)]
}, numeric(1))

arc <- function(p1, p2, bow, n = 80){
  d <- p2 - p1; L <- sqrt(sum(d^2)); if (L == 0) return(rbind(p1, p2))
  perp <- c(-d[2], d[1])/L; ctl <- (p1+p2)/2 + perp*bow*L
  t <- seq(0, 1, length.out = n)
  cbind((1-t)^2*p1[1] + 2*(1-t)*t*ctl[1] + t^2*p2[1],
        (1-t)^2*p1[2] + 2*(1-t)*t*ctl[2] + t^2*p2[2])
}

dfs   <- function(nd) if (is.null(kids[[nd]])) nd else unlist(lapply(kids[[nd]], dfs))
terms <- dfs(root); yt <- setNames(seq_along(terms), terms)
Yl <- list(); setY <- function(nd){
  y <- if (is.null(kids[[nd]])) yt[[nd]] else mean(vapply(kids[[nd]], setY, numeric(1)))
  Yl[[nd]] <<- y; y }
invisible(setY(root)); Y <- unlist(Yl)[nodes]

Xr <- setNames(vapply(nodes, function(c_) mean(ptc[clus==c_], na.rm=TRUE), numeric(1)), nodes)
bfs <- root; i <- 1
while (i <= length(bfs)) { if (!is.null(kids[[bfs[i]]])) bfs <- c(bfs, kids[[bfs[i]]]); i <- i+1 }
X <- Xr; for (nd in bfs[-1]) X[nd] <- max(Xr[nd], X[parent[[nd]]] + MIN_GAP)

lin_n <- vapply(paths, function(p) sum(N[p]), numeric(1)); main_i <- which.max(lin_n)
nbg <- setNames(rep("white", length(nodes)), nodes)
nbg[terms] <- TERM_COL; nbg[bpts] <- BP_COL; nbg[root] <- ROOT_COL
nfg <- ifelse(nbg %in% c(TERM_COL, ROOT_COL), "white", "grey10")

xd <- range(X); tk <- seq(floor(xd[1]/5)*5, ceiling(xd[2]/5)*5, by=5)
tk <- tk[tk >= xd[1]-1 & tk <= xd[2]+1]
lab0 <- max(X) + 3.0; xlim <- c(min(X) - 2.5, lab0 + 21); ybot <- 0.30

options(repr.plot.width = 13, repr.plot.height = 5.4)
layout(matrix(1:2, 1), widths = c(1.18, 1))

# ── LEFT · dendrogram ───────────────────────────────────────────────────────
par(mar = c(3.4, 0.8, 2.4, 0.4), xpd = NA)
plot(NA, xlim=xlim, ylim=c(ybot-0.35, length(terms)+1.0), xlab="", ylab="", axes=FALSE)
segments(min(tk), ybot, max(tk), ybot, col="grey55", lwd=0.8)
segments(tk, ybot, tk, ybot-0.09, col="grey55", lwd=0.8)
text(tk, ybot-0.26, tk, cex=0.64, col="grey35")
text(mean(range(tk)), ybot-0.52, "pseudotime", cex=0.7, col="grey35")
for (r in seq_len(nrow(E))) {
  f <- E$from[r]; t <- E$to[r]
  segments(X[f], Y[f], X[f], Y[t], col=ecol[r], lwd=elwd[r], lend=1)
  segments(X[f], Y[t], X[t], Y[t], col=ecol[r], lwd=elwd[r], lend=1)
}
points(X, Y, pch=21, bg=nbg[nodes], col="grey20", cex=NODE, lwd=1)
text(X, Y, nodes, font=2, cex=0.56, col=nfg[nodes])
for (k in seq_along(terms)) {
  t <- terms[k]; li <- which(vapply(paths, function(p) tail(p,1)==t, logical(1)))[1]
  segments(X[t]+0.8, Y[t], lab0-0.4, Y[t], col="grey90", lwd=0.4, lty=3)
  text(lab0, Y[t], sprintf("L%d%s", VALID[li], if (li==main_i) " \u25c0" else ""),
       adj=0, font=2, cex=0.68, col=LCOL[as.character(li)])
  text(lab0+5.2,  Y[t], sprintf("cl.%s", t), adj=0, cex=0.62, col="grey35")
  text(lab0+11.0, Y[t], format(N[t], big.mark=","), adj=1, cex=0.6, col="grey55")
}
text(lab0+11.0, length(terms)+0.75, "cells", adj=1, cex=0.55, col="grey60", font=3)
mtext("Branching structure", 3, line=0.55, adj=0, cex=0.92, font=2)
legend("topleft", inset=c(0.01,0.01), bty="n", ncol=2, cex=0.62, pt.cex=1.15,
       x.intersp=0.6, y.intersp=0.85, pch=21, col="grey20",
       pt.bg=c(ROOT_COL, BP_COL, TERM_COL, "white"),
       legend=c("root","branch","terminal","transit"))

# ── RIGHT · UMAP with curved haloed connectors ──────────────────────────────
dimred <- Embeddings(mg, DISP_RED)[,1:2]
cent <- t(vapply(nodes, function(i) colMeans(dimred[clus==i,,drop=FALSE]), numeric(2)))
rownames(cent) <- nodes

if (UMAP_BG == "pseudotime") {
  q <- pmin(pmax(ptc, quantile(ptc,.01,na.rm=TRUE)), quantile(ptc,.99,na.rm=TRUE))
  bgcol <- colorRampPalette(c("#EFEFEF","#D6E3EF","#B8CFE3"))(100)[cut(q,100,labels=FALSE)]
} else bgcol <- "grey90"

par(mar=c(3.4, 2.4, 2.4, 0.4), xpd=FALSE)
plot(dimred, col=bgcol, pch=16, cex=0.22, asp=1, axes=FALSE, xlab="", ylab="")

ord <- order(-E$flow)                       # trunk底 first, thin branches on top
for (r in ord) {                            # halo then line → readable crossings
  a <- arc(cent[E$from[r],], cent[E$to[r],], E$bow[r])
  lines(a, col="white",  lwd=elwd[r]*0.85 + 3.2, lend=1)
  lines(a, col=ecol[r],  lwd=elwd[r]*0.85,       lend=1)
}
key <- unique(c(root, bpts, terms))
points(cent[key,,drop=FALSE], pch=21, bg=nbg[key], col="grey15", cex=NODE*0.95, lwd=1.2)
text(cent[key,1], cent[key,2], key, font=2, cex=0.54, col=nfg[key])
transit <- setdiff(nodes, key)
if (length(transit)) {
  points(cent[transit,,drop=FALSE], pch=21, bg="white", col="grey45", cex=NODE*0.7, lwd=0.9)
  text(cent[transit,1], cent[transit,2], transit, font=2, cex=0.46, col="grey25")
}
mtext("UMAP", 3, line=0.55, adj=0, cex=0.92, font=2)
mtext("UMAP 1", 1, line=1.0, cex=0.66, col="grey45")
mtext("UMAP 2", 2, line=0.6, cex=0.66, col="grey45")

dev.copy(png, "mg_trajectory_publication.png", width=1950, height=810, res=150); dev.off()
dev.copy(svg, "mg_trajectory_publication.svg", width=13, height=5.4); dev.off()

cat(sprintf("main lineage: L%d (%s cells)\n", VALID[main_i], format(lin_n[main_i], big.mark=",")))
cat("branch points:", paste(sprintf("cl.%s\u2192{%s}", bpts,
    vapply(bpts, function(b) paste(kids[[b]], collapse=","), character(1))), collapse="  "), "\n")

---
## 08 · Trajectory × scores — post-hoc overlay

**Why.** Three UMAP panels on the same tree, coloured by score.

**The scores played no part in the fit.** This panel shows where they land on a trajectory built without them, which is the only reading that is not circular. Raw scores, not composites — see the note at the top.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) -> X_COL,
#       resolved from AXIS in the config cell.
# ════════════════════════════════════════════════════════════════════════════
# TRAJECTORY × SCORES — 3 UMAP panels, same tree, post-hoc score overlay
#   1 senescence · 2 homeo→DAM axis · 3 homeo→IRM axis
#   nodes: green = root · gold = branch · red = terminal (white-haloed)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; DISP_RED <- "umap.mg"
VALID <- c(1,2,3,5,6,7); NODE <- 2.1; BOW <- 0.16
ROOT_COL <- "#1B7837"; BP_COL <- "#E8B31F"; TERM_COL <- "#C0392B"
RIB <- "#3F3F3F"
DIV <- c("#2166AC","#7FB0D3","#F2F2F2","#E8A87C","#8C3A10")   # blue–grey–ochre

# ── ensure the homeo→state axes exist ───────────────────────────────────────
HOMEO_COL <- "Score_Homeostatic"
for (nm in c("DAM","IRM")) {
  az <- sprintf("%s_Homeo_axis_z", nm)
  src <- if (nm == "DAM") X_COL else "Score_IRM"
  if (!az %in% colnames(mg@meta.data)) {
    stopifnot(src %in% colnames(mg@meta.data), HOMEO_COL %in% colnames(mg@meta.data))
    raw <- mg@meta.data[[src]] - mg@meta.data[[HOMEO_COL]]
    mg[[sprintf("%s_Homeo_axis", nm)]] <- raw
    mg[[az]] <- as.numeric(scale(raw))
    cat("built", az, "\n")
  }
}

SCORES <- list(
  list(col="senescence_score", lab="Senescence score",        center=NA),
  list(col="DAM_Homeo_axis_z", lab="Homeo \u2192 DAM axis (z)", center=0),
  list(col="IRM_Homeo_axis_z", lab="Homeo \u2192 IRM axis (z)", center=0))

md <- mg@meta.data
ok <- vapply(SCORES, function(s) s$col %in% colnames(md), logical(1))
if (any(!ok)) cat("missing:", paste(vapply(SCORES[!ok], `[[`, "", "col"), collapse=", "), "\n")
SCORES <- SCORES[ok]

# ── tree structure from the fitted sce ──────────────────────────────────────
sds   <- SlingshotDataSet(sce)
paths <- lapply(slingLineages(sds), as.character)[VALID]
ptm   <- slingPseudotime(sds)
clus  <- as.character(md[[CLUSTER_LBL]])

E <- unique(do.call(rbind, lapply(paths, function(p)
       data.frame(from=p[-length(p)], to=p[-1], stringsAsFactors=FALSE))))
kids  <- split(E$to, E$from)
nodes <- unique(c(E$from, E$to)); root <- setdiff(E$from, E$to)[1]
bpts  <- names(kids)[lengths(kids) > 1]
terms <- unique(vapply(paths, function(p) tail(p,1), character(1)))
N     <- setNames(vapply(nodes, function(c_) sum(clus==c_), numeric(1)), nodes)
subN  <- function(nd){ ch <- kids[[nd]]
  if (is.null(ch)) N[[nd]] else N[[nd]] + sum(vapply(ch, subN, numeric(1))) }
fl    <- vapply(E$to, subN, numeric(1)); E$rel <- sqrt(fl/max(fl))
E$bow <- vapply(seq_len(nrow(E)), function(r){
  sib <- kids[[E$from[r]]]; k <- length(sib)
  if (k < 2) 0 else seq(-BOW, BOW, length.out=k)[match(E$to[r], sib)] }, numeric(1))

nbg <- setNames(rep("white", length(nodes)), nodes)
nbg[terms] <- TERM_COL; nbg[bpts] <- BP_COL; nbg[root] <- ROOT_COL
nfg <- ifelse(nbg %in% c(TERM_COL, ROOT_COL), "white", "grey10")

arc <- function(p1,p2,bow,n=70){ d <- p2-p1; L <- sqrt(sum(d^2))
  if (L==0) return(rbind(p1,p2)); pp <- c(-d[2],d[1])/L; ctl <- (p1+p2)/2 + pp*bow*L
  t <- seq(0,1,length.out=n)
  cbind((1-t)^2*p1[1]+2*(1-t)*t*ctl[1]+t^2*p2[1],
        (1-t)^2*p1[2]+2*(1-t)*t*ctl[2]+t^2*p2[2]) }
ribbon <- function(a,w1,w2){ tg <- rbind(a[2,]-a[1,], a[-1,]-a[-nrow(a),])
  L <- sqrt(rowSums(tg^2)); L[L==0] <- 1; tg <- tg/L
  nm <- cbind(-tg[,2], tg[,1]); w <- seq(w1,w2,length.out=nrow(a))/2
  rbind(a + nm*w, (a - nm*w)[nrow(a):1,]) }

dimred <- Embeddings(mg, DISP_RED)[,1:2]
cent <- t(vapply(nodes, function(i) colMeans(dimred[clus==i,,drop=FALSE]), numeric(2)))
rownames(cent) <- nodes
usc <- diff(range(dimred[,1]))/100
xr <- range(dimred[,1]); yr <- range(dimred[,2]); ord <- order(-E$rel)

draw_panel <- function(s){
  v  <- md[[s$col]]
  lo <- quantile(v, .02, na.rm=TRUE); hi <- quantile(v, .98, na.rm=TRUE)
  ct <- if (is.na(s$center)) median(v, na.rm=TRUE) else s$center
  h  <- max(ct - lo, hi - ct)                       # symmetric about centre
  brk <- seq(ct - h, ct + h, length.out=141)
  pal <- colorRampPalette(DIV)(140)
  ci  <- cut(pmin(pmax(v, ct-h), ct+h), brk, include.lowest=TRUE, labels=FALSE)
  ci[is.na(ci)] <- 70

  par(mar=c(3.2, 1.0, 2.6, 0.6), xpd=FALSE)
  plot(dimred, col=pal[ci], pch=16, cex=0.26, asp=1, axes=FALSE, xlab="", ylab="")
  for (r in ord) {
    a  <- arc(cent[E$from[r],], cent[E$to[r],], E$bow[r])
    w1 <- usc*(1.0 + 3.8*E$rel[r]); w2 <- w1*0.55
    polygon(ribbon(a, w1+usc*1.9, w2+usc*1.9), col="white", border=NA)
    polygon(ribbon(a, w1, w2), col=RIB, border=NA)
  }
  key <- unique(c(root, bpts, terms))
  points(cent[key,,drop=FALSE], pch=21, bg="white", col="white", cex=NODE*1.30, lwd=0)
  points(cent[key,,drop=FALSE], pch=21, bg=nbg[key], col="grey10", cex=NODE, lwd=1.1)
  text(cent[key,1], cent[key,2], key, font=2, cex=0.5, col=nfg[key])

  bx <- seq(xr[1]+.03*diff(xr), xr[1]+.30*diff(xr), length.out=141)
  for (j in 1:140) rect(bx[j], yr[1]-.005*diff(yr), bx[j+1], yr[1]+.026*diff(yr),
                        col=pal[j], border=NA)
  rect(bx[1], yr[1]-.005*diff(yr), bx[141], yr[1]+.026*diff(yr), border="grey60", lwd=0.5)
  text(bx[1],   yr[1]-.042*diff(yr), sprintf("%.2f", ct-h), cex=0.52, adj=0, col="grey35")
  text(bx[141], yr[1]-.042*diff(yr), sprintf("%.2f", ct+h), cex=0.52, adj=1, col="grey35")
  mtext(s$lab, 3, line=0.6, adj=0, cex=0.86, font=2)
}

options(repr.plot.width = 15, repr.plot.height = 5.2)
layout(matrix(seq_along(SCORES), 1))
invisible(lapply(SCORES, draw_panel))
legend("bottomright", bty="n", cex=0.58, pt.cex=1.1, x.intersp=0.7, y.intersp=0.9,
       pch=21, col="grey10", pt.bg=c(ROOT_COL, BP_COL, TERM_COL),
       legend=c("root","branch","terminal"))

dev.copy(png, "mg_trajectory_3scores.png", width=2250, height=780, res=150); dev.off()
dev.copy(svg, "mg_trajectory_3scores.svg", width=15, height=5.2); dev.off()

# ── quantitative companion ──────────────────────────────────────────────────
cat("\n════ Spearman rho: score vs pseudotime, per lineage ════\n")
res <- do.call(rbind, lapply(seq_along(VALID), function(k){
  i <- VALID[k]; f <- is.finite(ptm[, i])
  r <- vapply(SCORES, function(s)
        suppressWarnings(cor(ptm[f,i], md[[s$col]][f], method="spearman")), numeric(1))
  data.frame(L=i, terminal=paste0("cl.", tail(paths[[k]],1)), n=sum(f),
             setNames(as.list(round(r,3)),
                      c("sen","DAM_axis","IRM_axis")[seq_along(SCORES)]),
             check.names=FALSE)
}))
print(res, row.names=FALSE)
cat("\nrho>0 = rises toward terminal | ~0 = independent of trajectory\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) -> X_COL,
#       resolved from AXIS in the config cell.
# ════════════════════════════════════════════════════════════════════════════
# TRAJECTORY × SCORES — 3 UMAP panels, same tree, post-hoc score overlay
#   1 senescence · 2 homeo→DAM axis · 3 homeo→IRM axis
#   connectors: curved stroked lines + white halo, lwd ∝ cells downstream
#   nodes: green = root · gold = branch · red = terminal
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; DISP_RED <- "umap.mg"
VALID    <- c(1,2,3,5,6,7)
NODE     <- 2.1
BOW      <- 0.20                      # edge curvature (0 = straight)
EDGE_COL <- "grey20"                  # neutral so the score gradient reads; see note
ROOT_COL <- "#1B7837"; BP_COL <- "#E8B31F"; TERM_COL <- "#C0392B"
DIV      <- c("#2166AC","#7FB0D3","#F2F2F2","#E8A87C","#8C3A10")

# ── ensure the homeo→state axes exist ───────────────────────────────────────
HOMEO_COL <- "Score_Homeostatic"
for (nm in c("DAM","IRM")) {
  az  <- sprintf("%s_Homeo_axis_z", nm)
  src <- if (nm == "DAM") X_COL else "Score_IRM"
  if (!az %in% colnames(mg@meta.data)) {
    stopifnot(src %in% colnames(mg@meta.data), HOMEO_COL %in% colnames(mg@meta.data))
    raw <- mg@meta.data[[src]] - mg@meta.data[[HOMEO_COL]]
    mg[[sprintf("%s_Homeo_axis", nm)]] <- raw
    mg[[az]] <- as.numeric(scale(raw))
    cat("built", az, "\n")
  }
}

SCORES <- list(
  list(col="senescence_score", lab="Senescence score",         center=NA),
  list(col="DAM_Homeo_axis_z", lab="Homeo \u2192 DAM axis (z)", center=0),
  list(col="IRM_Homeo_axis_z", lab="Homeo \u2192 IRM axis (z)", center=0))

md <- mg@meta.data
ok <- vapply(SCORES, function(s) s$col %in% colnames(md), logical(1))
if (any(!ok)) cat("missing:", paste(vapply(SCORES[!ok], `[[`, "", "col"), collapse=", "), "\n")
SCORES <- SCORES[ok]

# ── tree structure from the fitted sce ──────────────────────────────────────
sds   <- SlingshotDataSet(sce)
paths <- lapply(slingLineages(sds), as.character)[VALID]
ptm   <- slingPseudotime(sds)
clus  <- as.character(md[[CLUSTER_LBL]])

E <- unique(do.call(rbind, lapply(paths, function(p)
       data.frame(from=p[-length(p)], to=p[-1], stringsAsFactors=FALSE))))
kids  <- split(E$to, E$from)
nodes <- unique(c(E$from, E$to)); root <- setdiff(E$from, E$to)[1]
bpts  <- names(kids)[lengths(kids) > 1]
terms <- unique(vapply(paths, function(p) tail(p,1), character(1)))
N     <- setNames(vapply(nodes, function(c_) sum(clus==c_), numeric(1)), nodes)

subN  <- function(nd){ ch <- kids[[nd]]
  if (is.null(ch)) N[[nd]] else N[[nd]] + sum(vapply(ch, subN, numeric(1))) }
E$flow <- vapply(E$to, subN, numeric(1))
elwd   <- 0.9 + 5.0 * sqrt(E$flow / max(E$flow))          # same law as the 2-panel fig
E$bow  <- vapply(seq_len(nrow(E)), function(r){
  sib <- kids[[E$from[r]]]; k <- length(sib)
  if (k < 2) return(0)
  seq(-BOW, BOW, length.out = k)[match(E$to[r], sib)] }, numeric(1))

nbg <- setNames(rep("white", length(nodes)), nodes)
nbg[terms] <- TERM_COL; nbg[bpts] <- BP_COL; nbg[root] <- ROOT_COL
nfg <- ifelse(nbg %in% c(TERM_COL, ROOT_COL), "white", "grey10")

arc <- function(p1, p2, bow, n = 80){
  d <- p2 - p1; L <- sqrt(sum(d^2)); if (L == 0) return(rbind(p1, p2))
  perp <- c(-d[2], d[1])/L; ctl <- (p1+p2)/2 + perp*bow*L
  t <- seq(0, 1, length.out = n)
  cbind((1-t)^2*p1[1] + 2*(1-t)*t*ctl[1] + t^2*p2[1],
        (1-t)^2*p1[2] + 2*(1-t)*t*ctl[2] + t^2*p2[2])
}

dimred <- Embeddings(mg, DISP_RED)[,1:2]
cent <- t(vapply(nodes, function(i) colMeans(dimred[clus==i,,drop=FALSE]), numeric(2)))
rownames(cent) <- nodes
xr <- range(dimred[,1]); yr <- range(dimred[,2])
ord <- order(-E$flow)                                     # thick first, thin on top

draw_panel <- function(s){
  v  <- md[[s$col]]
  lo <- quantile(v, .02, na.rm=TRUE); hi <- quantile(v, .98, na.rm=TRUE)
  ct <- if (is.na(s$center)) median(v, na.rm=TRUE) else s$center
  h  <- max(ct - lo, hi - ct)
  brk <- seq(ct - h, ct + h, length.out=141)
  pal <- colorRampPalette(DIV)(140)
  ci  <- cut(pmin(pmax(v, ct-h), ct+h), brk, include.lowest=TRUE, labels=FALSE)
  ci[is.na(ci)] <- 70

  par(mar=c(3.2, 1.0, 2.6, 0.6), xpd=FALSE)
  plot(dimred, col=pal[ci], pch=16, cex=0.8, asp=1, axes=FALSE, xlab="", ylab="")

  for (r in ord) {                                        # halo then stroke
    a <- arc(cent[E$from[r],], cent[E$to[r],], E$bow[r])
    lines(a, col="white",    lwd=elwd[r]*0.85 + 3.2, lend=1)
    lines(a, col=EDGE_COL,   lwd=elwd[r]*0.85,       lend=1)
  }

  key <- unique(c(root, bpts, terms))
  points(cent[key,,drop=FALSE], pch=21, bg="white", col="white", cex=NODE*1.28, lwd=0)
  points(cent[key,,drop=FALSE], pch=21, bg=nbg[key], col="grey10", cex=NODE, lwd=1.1)
  text(cent[key,1], cent[key,2], key, font=2, cex=0.5, col=nfg[key])

  bx <- seq(xr[1]+.03*diff(xr), xr[1]+.30*diff(xr), length.out=141)
  for (j in 1:140) rect(bx[j], yr[1]-.005*diff(yr), bx[j+1], yr[1]+.026*diff(yr),
                        col=pal[j], border=NA)
  rect(bx[1], yr[1]-.005*diff(yr), bx[141], yr[1]+.026*diff(yr), border="grey60", lwd=0.5)
  text(bx[1],   yr[1]-.042*diff(yr), sprintf("%.2f", ct-h), cex=0.52, adj=0, col="grey35")
  text(bx[141], yr[1]-.042*diff(yr), sprintf("%.2f", ct+h), cex=0.52, adj=1, col="grey35")
  mtext(s$lab, 3, line=0.6, adj=0, cex=0.86, font=2)
}

options(repr.plot.width = 15, repr.plot.height = 5.2)
layout(matrix(seq_along(SCORES), 1))
invisible(lapply(SCORES, draw_panel))
legend("bottomright", bty="n", cex=0.58, pt.cex=1.1, x.intersp=0.7, y.intersp=0.9,
       pch=21, col="grey10", pt.bg=c(ROOT_COL, BP_COL, TERM_COL),
       legend=c("root","branch","terminal"))

dev.copy(png, "mg_trajectory_3scores.png", width=2250, height=780, res=150); dev.off()
dev.copy(svg, "mg_trajectory_3scores.svg", width=15, height=5.2); dev.off()

# ── quantitative companion ──────────────────────────────────────────────────
cat("\n════ Spearman rho: score vs pseudotime, per lineage ════\n")
res <- do.call(rbind, lapply(seq_along(VALID), function(k){
  i <- VALID[k]; f <- is.finite(ptm[, i])
  r <- vapply(SCORES, function(s)
        suppressWarnings(cor(ptm[f,i], md[[s$col]][f], method="spearman")), numeric(1))
  data.frame(L=i, terminal=paste0("cl.", tail(paths[[k]],1)), n=sum(f),
             setNames(as.list(round(r,3)),
                      c("sen","DAM_axis","IRM_axis")[seq_along(SCORES)]),
             check.names=FALSE)
}))
print(res, row.names=FALSE)
cat("\nrho>0 = rises toward terminal | ~0 = independent of trajectory\n")

---
## 09 · Trajectory × state

**Why.** The same tree coloured by the module 09 state label. State and pseudotime agreeing is meaningful because neither informed the other: the states came from panel scores, the trajectory from clusters and the manifold.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# TRAJECTORY × STATE — UMAP coloured by microglial state label + connectors
#   cells : canonical state palette (lightened) · nodes: green/gold/red on top
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment) })

CLUSTER_LBL <- "mg_cluster"; STATE_COL <- "microglia_state"; DISP_RED <- "umap.mg"
VALID    <- c(1,2,3,5,6,7)
NODE     <- 2.3
BOW      <- 0.20
EDGE_COL <- "grey15"
CELL_CEX <- 0.8
CELL_ALPHA <- 0.45
NODE_STYLE <- "colour"                # "colour" (green/gold/red) or "shape"
ROOT_COL <- "#1B7837"; BP_COL <- "#E8B31F"; TERM_COL <- "#C0392B"
STATE_ORDER  <- c("Homeostatic","ARM","IRM","Stress","DAM_like")
STATE_COLORS <- c(Homeostatic="#7F8C8D", ARM="#E67E22", IRM="#2980B9",
                  Stress="#8E44AD", DAM_like="#C0392B")

md   <- mg@meta.data
stopifnot(STATE_COL %in% colnames(md))
st   <- factor(as.character(md[[STATE_COL]]), levels = STATE_ORDER)
clus <- as.character(md[[CLUSTER_LBL]])

# ── tree structure from the fitted sce ──────────────────────────────────────
sds   <- SlingshotDataSet(sce)
paths <- lapply(slingLineages(sds), as.character)[VALID]
ptm   <- slingPseudotime(sds)

E <- unique(do.call(rbind, lapply(paths, function(p)
       data.frame(from=p[-length(p)], to=p[-1], stringsAsFactors=FALSE))))
kids  <- split(E$to, E$from)
nodes <- unique(c(E$from, E$to)); root <- setdiff(E$from, E$to)[1]
bpts  <- names(kids)[lengths(kids) > 1]
terms <- unique(vapply(paths, function(p) tail(p,1), character(1)))
N     <- setNames(vapply(nodes, function(c_) sum(clus==c_), numeric(1)), nodes)

subN  <- function(nd){ ch <- kids[[nd]]
  if (is.null(ch)) N[[nd]] else N[[nd]] + sum(vapply(ch, subN, numeric(1))) }
E$flow <- vapply(E$to, subN, numeric(1))
elwd   <- 0.9 + 5.0 * sqrt(E$flow / max(E$flow))
E$bow  <- vapply(seq_len(nrow(E)), function(r){
  sib <- kids[[E$from[r]]]; k <- length(sib)
  if (k < 2) return(0)
  seq(-BOW, BOW, length.out = k)[match(E$to[r], sib)] }, numeric(1))

arc <- function(p1, p2, bow, n = 80){
  d <- p2 - p1; L <- sqrt(sum(d^2)); if (L == 0) return(rbind(p1, p2))
  perp <- c(-d[2], d[1])/L; ctl <- (p1+p2)/2 + perp*bow*L
  t <- seq(0, 1, length.out = n)
  cbind((1-t)^2*p1[1] + 2*(1-t)*t*ctl[1] + t^2*p2[1],
        (1-t)^2*p1[2] + 2*(1-t)*t*ctl[2] + t^2*p2[2]) }

dimred <- Embeddings(mg, DISP_RED)[,1:2]
cent <- t(vapply(nodes, function(i) colMeans(dimred[clus==i,,drop=FALSE]), numeric(2)))
rownames(cent) <- nodes
ord <- order(-E$flow)

# ── node styling ────────────────────────────────────────────────────────────
key <- unique(c(root, bpts, terms))
if (NODE_STYLE == "colour") {
  nbg <- setNames(rep("white", length(nodes)), nodes)
  nbg[terms] <- TERM_COL; nbg[bpts] <- BP_COL; nbg[root] <- ROOT_COL
  npch <- setNames(rep(21, length(nodes)), nodes)
} else {
  nbg  <- setNames(rep("white", length(nodes)), nodes); nbg[key] <- "grey15"
  npch <- setNames(rep(21, length(nodes)), nodes)
  npch[bpts] <- 22; npch[root] <- 23
}
nfg <- ifelse(nbg %in% c(TERM_COL, ROOT_COL, "grey15"), "white", "grey10")

# ── plot ────────────────────────────────────────────────────────────────────
o <- sample(nrow(dimred))                       # shuffle so no state sits on top
cellcol <- adjustcolor(STATE_COLORS[as.character(st)], alpha.f = CELL_ALPHA)
cellcol[is.na(st)] <- adjustcolor("grey80", alpha.f = CELL_ALPHA)

options(repr.plot.width = 8.4, repr.plot.height = 7.4)
par(mar = c(3.0, 2.6, 2.6, 1.2), xpd = FALSE)
plot(dimred[o,], col = cellcol[o], pch = 16, cex = CELL_CEX, asp = 1,
     axes = FALSE, xlab = "", ylab = "")

for (r in ord) {
  a <- arc(cent[E$from[r],], cent[E$to[r],], E$bow[r])
  lines(a, col = "white",  lwd = elwd[r]*0.85 + 3.4, lend = 1)
  lines(a, col = EDGE_COL, lwd = elwd[r]*0.85,       lend = 1)
}
points(cent[key,,drop=FALSE], pch = 21, bg = "white", col = "white",
       cex = NODE*1.35, lwd = 0)
points(cent[key,,drop=FALSE], pch = npch[key], bg = nbg[key], col = "grey10",
       cex = NODE, lwd = 1.2)
text(cent[key,1], cent[key,2], key, font = 2, cex = 0.54, col = nfg[key])

transit <- setdiff(nodes, key)
if (length(transit)) {
  points(cent[transit,,drop=FALSE], pch = 21, bg = "white", col = "grey40",
         cex = NODE*0.68, lwd = 0.9)
  text(cent[transit,1], cent[transit,2], transit, font = 2, cex = 0.44, col = "grey25")
}

mtext("Microglial trajectory \u00b7 cells coloured by state", 3, line = 0.7,
      adj = 0, cex = 0.98, font = 2)
mtext("UMAP 1", 1, line = 1.0, cex = 0.7, col = "grey45")
mtext("UMAP 2", 2, line = 0.7, cex = 0.7, col = "grey45")

# ── ONE combined legend: states + node roles (blank row separates blocks) ────
par(xpd = NA)
ns <- length(STATE_ORDER)
if (NODE_STYLE == "colour") {
  legend("topright", bty = "n", cex = 0.68, pt.cex = 1.3, y.intersp = 0.92,
         pch   = c(rep(21, ns), NA, 21, 21, 21),
         pt.bg = c(STATE_COLORS[STATE_ORDER], NA, ROOT_COL, BP_COL, TERM_COL),
         col   = c(rep("grey35", ns), NA, rep("grey10", 3)),
         legend = c(STATE_ORDER, "", "root", "branch", "terminal"))
} else {
  legend("topright", bty = "n", cex = 0.68, pt.cex = 1.3, y.intersp = 0.92,
         pch   = c(rep(21, ns), NA, 23, 22, 21),
         pt.bg = c(STATE_COLORS[STATE_ORDER], NA, rep("grey15", 3)),
         col   = c(rep("grey35", ns), NA, rep("grey10", 3)),
         legend = c(STATE_ORDER, "", "root", "branch", "terminal"))
}

dev.copy(png, "mg_trajectory_by_state.png", width = 1260, height = 1110, res = 150); dev.off()
dev.copy(svg, "mg_trajectory_by_state.svg", width = 8.4, height = 7.4); dev.off()

# ── which state dominates each node? ────────────────────────────────────────
cat("\n════ dominant state per cluster ════\n")
tab <- table(clus, st)
dom <- data.frame(
  cluster = nodes,
  role    = ifelse(nodes == root, "root",
             ifelse(nodes %in% bpts, "branch",
             ifelse(nodes %in% terms, "terminal", "transit"))),
  n       = as.integer(N[nodes]),
  top     = STATE_ORDER[apply(tab[nodes, , drop=FALSE], 1, which.max)],
  pct     = round(100*apply(tab[nodes, , drop=FALSE], 1,
                            function(x) max(x)/sum(x)), 1))
print(dom[order(match(dom$role, c("root","branch","terminal","transit"))), ],
      row.names = FALSE)
cat("\n════ state composition of terminals (%) ════\n")
print(round(100*prop.table(tab[terms, , drop=FALSE], 1), 1))

---
## 10 · Cell density along pseudotime

**Why.** Whether the groups occupy the same trajectory at different densities or different parts of it. Both are compatible with the same mean pseudotime, so the mean alone cannot distinguish them.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL DENSITY ALONG PSEUDOTIME — split by group (shared trajectory)
#   asks: do AD cells pile toward the activated end vs Control?
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr) })

gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
df <- mg_s@meta.data
df$pseudotime <- mg_s$pseudotime
df$grp2 <- factor(unname(gmap[as.character(df$Study_Group)]), levels=c("Control","AD"))
df <- df[!is.na(df$grp2) & is.finite(df$pseudotime), ]

RED <- "#C0392B"; BLUE <- "#3D7C9A"

# ---- DENSITY PLOT ----------------------------------------------------------
p_dens <- ggplot(df, aes(pseudotime, fill=grp2, color=grp2)) +
    geom_density(alpha=0.30, linewidth=0.9, adjust=1.2) +
    scale_fill_manual(values=c(Control=BLUE, AD=RED), name=NULL) +
    scale_color_manual(values=c(Control=BLUE, AD=RED), name=NULL) +
    labs(title="Cell density along the activation trajectory, by group",
         subtitle="Shared pseudotime · resting (low) → activated (high) · shift right = more activated cells",
         x="pseudotime  (resting → activated)", y="cell density") +
    theme_classic(base_size=11) +
    theme(plot.title=element_text(size=11, face="bold"),
          plot.subtitle=element_text(size=8, color="grey45"),
          legend.position="top",
          panel.border=element_rect(color="black", fill=NA, linewidth=0.4))
options(repr.plot.width=7, repr.plot.height=4.5); print(p_dens)
if (exists("save_figure")) save_figure(p_dens, "mg_pseudotime_density_byGroup", width=7, height=4.5)

# ---- HONEST TEST: is the group shift real at the DONOR level? ---------------
# each donor's MEDIAN pseudotime; compare AD vs Control across donors (not cells)
donor_pt <- df %>% group_by(Donor, grp2) %>%
    summarise(median_pt = median(pseudotime), n_cells = n(), .groups="drop") %>%
    filter(n_cells >= 20)                       # donors with enough cells
cat("\n════ donors per group (≥20 cells) ════\n"); print(table(donor_pt$grp2))
cat("\n════ donor-level median pseudotime, AD vs Control ════\n")
print(donor_pt %>% group_by(grp2) %>%
        summarise(med_of_medians = median(median_pt), IQR = IQR(median_pt)))
cat("\nWilcoxon (donor-level):\n")
print(wilcox.test(median_pt ~ grp2, data = donor_pt))

# fraction of each group's cells in the "activated" half (above global median)
gm <- median(df$pseudotime)
cat(sprintf("\n════ %% cells past global median (%.1f) = 'activated half' ════\n", gm))
print(df %>% group_by(grp2) %>% summarise(pct_activated = round(100*mean(pseudotime > gm),1)))

cat("\n✓ density-by-group done.\n")

In [ ]:
donor_frac <- df %>% group_by(Donor, grp2) %>%
    summarise(pct_act = mean(pseudotime > median(df$pseudotime)), n=n(), .groups="drop") %>%
    filter(n >= 20)
wilcox.test(pct_act ~ grp2, data = donor_frac)

---
## 11 · Progression panels

**Why.** Donor-ordered traces — activation, senescence, SASP, %SnC and the clinical score on one axis, so the order in which they rise can be read.

**Ordering is group → Braak → CDR.** This is a *donor* ordering, not a fitted trajectory, and it is descriptive: it shows which traces peak earlier along a clinical progression. It does not establish that one causes another.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PROGRESSION PANEL (fresh) — NA→0 on clinical scores, sort group → Braak → CDR
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
`%||%` <- function(a, b) if (is.null(a)) b else a
na_fb  <- function(x, fb) { x[is.na(x)] <- fb; x }

# ─── (1) NA → 0 on clinical/pathology scores ───
PATH_COLS <- intersect(c("Braak","CDR"), colnames(prog))
for (c_ in PATH_COLS) {
    n_na <- sum(is.na(prog[[c_]]))
    prog[[c_]][is.na(prog[[c_]])] <- 0
    cat(sprintf("  %s: %d NA → 0\n", c_, n_na))
}

# ─── (2) sort: group → Braak → CDR (no NAs now) ───
SORT_PATH <- "Braak"; SORT_TIE <- "CDR"
plot_df <- prog
plot_df$Study_Group <- factor(plot_df$Study_Group, levels = sg_ordered)
plot_df <- plot_df[order(plot_df$Study_Group, plot_df[[SORT_PATH]], plot_df[[SORT_TIE]]), ]
plot_df$x <- seq_len(nrow(plot_df)); rownames(plot_df) <- NULL
cat(sprintf("  sorted Study_Group → %s → %s | %d donors\n", SORT_PATH, SORT_TIE, nrow(plot_df)))

# ─── (3) z-scores + smoothed traces ───
z_score <- function(v){ v<-as.numeric(v); if(sum(!is.na(v))==0) return(v)
    s<-sd(v,na.rm=TRUE); if(s==0) return(v-mean(v,na.rm=TRUE)); (v-mean(v,na.rm=TRUE))/s }
smooth_trace <- function(y, wf=0.10, mn=5, mx=21){ y<-as.numeric(y); n<-length(y)
    if(n<3||sum(!is.na(y))<3) return(y); w<-max(mn,min(mx,round(n*wf))); if(w%%2==0) w<-w+1
    h<-(w-1)%/%2; out<-numeric(n); for(i in seq_len(n)){lo<-max(1,i-h);hi<-min(n,i+h);out[i]<-mean(y[lo:hi],na.rm=TRUE)}
    out[is.nan(out)]<-NA; out }

modules_in_prog   <- intersect(PROGRESSION_MODULES, gsub("^mean_","", grep("^mean_", colnames(prog), value=TRUE)))
available_markers <- CANONICAL_MARKER_ORDER[CANONICAL_MARKER_ORDER %in% colnames(prog)]

plot_df$pct_sen_z <- z_score(plot_df$pct_sen)
for (mod in modules_in_prog) plot_df[[paste0("mean_",mod,"_z")]] <- z_score(plot_df[[paste0("mean_",mod)]])
for (m in available_markers)  plot_df[[paste0(m,"_z")]]          <- z_score(plot_df[[m]])

trace_rows <- list()
for (m in available_markers) trace_rows[[paste0("p_",m)]] <- data.frame(
    x=plot_df$x, y=smooth_trace(plot_df[[paste0(m,"_z")]]), trace=MARKER_LABELS[m],
    color=MARKER_COLORS[m], linetype=ifelse(m %in% INVERSE_MARKERS,"dashed","solid"),
    size=0.8, group="pathology", legord=match(m,CANONICAL_MARKER_ORDER), stringsAsFactors=FALSE)
for (mod in modules_in_prog) trace_rows[[paste0("m_",mod)]] <- data.frame(
    x=plot_df$x, y=smooth_trace(plot_df[[paste0("mean_",mod,"_z")]]), trace=mod,
    color=na_fb(MODULE_TRACE_COLORS[mod], fallback_color), linetype="solid",
    size=1.1, group="module", legord=100+match(mod,modules_in_prog), stringsAsFactors=FALSE)
y_snc <- smooth_trace(plot_df$pct_sen_z)
trace_rows[["snc"]] <- data.frame(x=plot_df$x, y=y_snc, trace="%SnC",
    color=MODULE_TRACE_COLORS[["%SnC"]], linetype="solid", size=1.6, group="snc", legord=999, stringsAsFactors=FALSE)
trace_df <- do.call(rbind, trace_rows); rownames(trace_df) <- NULL
trace_df$trace <- factor(trace_df$trace, levels=unique(trace_df$trace[order(trace_df$legord)]))

# ─── (4) bands / labels / dividers / dots ───
band_rows<-list(); label_rows<-list(); divider_x<-numeric(0)
for (sg in sg_ordered){ d<-plot_df[plot_df$Study_Group==sg,]; if(nrow(d)==0) next
    x0<-min(d$x)-0.5; x1<-max(d$x)+0.5; bc<-na_fb(STUDY_GROUP_PALETTE[sg], fallback_color)
    band_rows[[sg]]<-data.frame(xmin=x0,xmax=x1,ymin=-Inf,ymax=Inf,fill=bc,stringsAsFactors=FALSE)
    label_rows[[sg]]<-data.frame(x=(x0+x1)/2,label_main=sg,label_n=sprintf("n=%d",nrow(d)),color=bc,stringsAsFactors=FALSE)}
band_df<-do.call(rbind,band_rows); label_df<-do.call(rbind,label_rows)
for (i in seq_along(sg_ordered)[-length(sg_ordered)]){ d<-plot_df[plot_df$Study_Group==sg_ordered[i],]
    if(nrow(d)>0) divider_x<-c(divider_x,max(d$x)+0.5)}
dot_df<-plot_df[,c("x","Study_Group")]; dot_df$y<-y_snc
dot_df$color<-na_fb(STUDY_GROUP_PALETTE[as.character(dot_df$Study_Group)], fallback_color)
dot_df<-dot_df[!is.na(dot_df$y),]

y_range<-range(trace_df$y,na.rm=TRUE); y_pad<-0.12*diff(y_range); y_lim<-c(y_range[1]-y_pad,y_range[2]+y_pad)

# ─── (5) main panel ───
p_overall <- ggplot() +
    geom_rect(data=band_df, aes(xmin=xmin,xmax=xmax,ymin=ymin,ymax=ymax,fill=fill), alpha=0.12, inherit.aes=FALSE) +
    scale_fill_identity() +
    {if(length(divider_x)>0) geom_vline(xintercept=divider_x, color="grey60", linetype="dotted", linewidth=0.4)} +
    geom_hline(yintercept=0, color="grey60", linetype="dashed", linewidth=0.4) +
    geom_line(data=trace_df[trace_df$group %in% c("pathology","module"),], aes(x=x,y=y,group=trace),
              color="white", linewidth=2.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[trace_df$group %in% c("pathology","module"),],
              aes(x=x,y=y,color=color,linetype=linetype,group=trace,linewidth=size)) +
    geom_line(data=trace_df[trace_df$group=="snc",], aes(x=x,y=y), color="white", linewidth=3.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[trace_df$group=="snc",], aes(x=x,y=y), color=MODULE_TRACE_COLORS[["%SnC"]], linewidth=1.8, lineend="round") +
    geom_point(data=dot_df, aes(x=x,y=y,fill=color), shape=21, color="black", size=1.7, stroke=0.3) +
    scale_color_identity()+scale_linetype_identity()+scale_linewidth_identity() +
    scale_x_continuous(expand=c(0,0)) + scale_y_continuous(limits=y_lim) +
    labs(title=sprintf("Progression: %s | %s | group → Braak → CDR | n=%d", CELL_TYPE, DATASET, nrow(plot_df)),
         x="Donors (group → Braak → CDR)", y="z-score") +
    theme_minimal(base_size=10) +
    theme(plot.title=element_text(size=11,face="bold",color="#333333",hjust=0,margin=margin(b=6)),
          plot.title.position="plot", plot.background=element_rect(fill="white",color=NA),
          panel.border=element_rect(color="#333333",fill=NA,linewidth=0.6),
          panel.grid.major.y=element_line(color="grey92",linewidth=0.3), panel.grid.minor=element_blank(),
          panel.grid.major.x=element_blank(), axis.text.x=element_blank(), axis.ticks.x=element_blank(),
          axis.title.x=element_text(size=8.5,color="#666666",margin=margin(t=6)),
          axis.text.y=element_text(size=9,color="#333333"), axis.title.y=element_text(size=9.5,color="#333333"),
          legend.position="none", plot.margin=margin(8,8,4,8))

# ─── (6) strip + legend ───
strip_plot <- ggplot(label_df) +
    geom_text(aes(x=x,y=1,label=label_main,color=color), size=2.7, fontface="bold", vjust=1) +
    geom_text(aes(x=x,y=0.45,label=label_n,color=color), size=2.4, vjust=1) +
    scale_color_identity() + scale_x_continuous(limits=c(0.5,nrow(plot_df)+0.5),expand=c(0,0)) +
    scale_y_continuous(limits=c(0,1.1)) + theme_void() +
    theme(plot.margin=margin(2,8,0,8), plot.background=element_rect(fill="white",color=NA))

legend_items<-list()
for (m in available_markers) legend_items[[m]]<-data.frame(label=MARKER_LABELS[m], color=MARKER_COLORS[m],
    linetype=ifelse(m %in% INVERSE_MARKERS,"dashed","solid"), bold=FALSE, stringsAsFactors=FALSE)
for (mod in modules_in_prog) legend_items[[mod]]<-data.frame(label=mod, color=na_fb(MODULE_TRACE_COLORS[mod],fallback_color),
    linetype="solid", bold=FALSE, stringsAsFactors=FALSE)
legend_items[["snc"]]<-data.frame(label="%SnC", color=MODULE_TRACE_COLORS[["%SnC"]], linetype="solid", bold=TRUE, stringsAsFactors=FALSE)
leg_df<-do.call(rbind,legend_items); rownames(leg_df)<-NULL
n_items<-nrow(leg_df); leg_df$slot<-seq_len(n_items)
leg_df$x_seg_lo<-leg_df$slot-0.5+0.05; leg_df$x_seg_hi<-leg_df$x_seg_lo+0.30; leg_df$x_text<-leg_df$x_seg_hi+0.05
legend_plot <- ggplot(leg_df) +
    geom_segment(aes(x=x_seg_lo,xend=x_seg_hi,y=1,yend=1,color=color,linetype=linetype), linewidth=0.9, lineend="round") +
    geom_text(aes(x=x_text,y=1,label=label,color=color,fontface=ifelse(bold,"bold","plain")), size=2.7, hjust=0, vjust=0.5) +
    scale_color_identity()+scale_linetype_identity() +
    scale_x_continuous(limits=c(0.4,n_items+0.6),expand=c(0,0)) + scale_y_continuous(limits=c(0.5,1.5),expand=c(0,0)) +
    theme_void() + theme(plot.margin=margin(2,8,4,8), plot.background=element_rect(fill="white",color=NA))

# ─── (7) compose + save ───
composed <- strip_plot / p_overall / legend_plot + plot_layout(heights=c(0.10,1,0.08))
save_figure(composed, slug=sprintf("%s_progression_braak_na0", CELL_TYPE), width=9.5, height=4.6)
options(repr.plot.width=10, repr.plot.height=4.9); print(composed)

# ─── (8) peak diagnostics ───
cat("\nsmoothed trace peaks:\n")
rp<-function(nm,v){sm<-smooth_trace(v); if(all(is.na(sm))){cat(sprintf("  %-18s all NaN\n",nm));return()}
    pk<-which.max(sm); cat(sprintf("  %-18s peak idx=%3d group=%s z=%+.2f\n",nm,pk,as.character(plot_df$Study_Group[pk]),sm[pk]))}
rp("%SnC", plot_df$pct_sen_z); for(mod in modules_in_prog) rp(mod, plot_df[[paste0("mean_",mod,"_z")]])

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# AGGREGATE IFN + DAM + Homeostatic per donor into prog (cell-level in mg)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages(library(dplyr))

# confirm the cell-level score columns exist
for (c_ in c("Score_IRM","DAM_Homeo_axis_z","Score_Homeostatic"))
    stopifnot(c_ %in% colnames(mg@meta.data))

# per-donor means of the cell-level scores
donor_scores <- mg@meta.data %>%
    group_by(Donor) %>%
    summarise(mean_IFN   = mean(Score_IRM,         na.rm=TRUE),
              mean_DAM   = mean(DAM_Homeo_axis_z,   na.rm=TRUE),
              mean_Homeo = mean(Score_Homeostatic,  na.rm=TRUE),
              .groups="drop")

# drop any pre-existing copies so re-running doesn't create .x/.y duplicates
prog <- prog[, !colnames(prog) %in% c("mean_IFN","mean_DAM","mean_Homeo")]

# merge into prog (matches on Donor)
prog <- dplyr::left_join(prog, donor_scores, by = "Donor")
cat("added to prog: mean_IFN, mean_DAM, mean_Homeo\n")
cat("NA after merge — IFN:", sum(is.na(prog$mean_IFN)),
    "| DAM:",   sum(is.na(prog$mean_DAM)),
    "| Homeo:", sum(is.na(prog$mean_Homeo)), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PROGRESSION — 5 traces: IFN-z, DAM-z, SASP-z, %SnC, CDR-z | group→Braak→CDR
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
`%||%` <- function(a,b) if (is.null(a)) b else a
na_fb  <- function(x,fb){ x[is.na(x)]<-fb; x }
z_score <- function(v){ v<-as.numeric(v); if(sum(!is.na(v))==0) return(v)
    s<-sd(v,na.rm=TRUE); if(s==0) return(v-mean(v,na.rm=TRUE)); (v-mean(v,na.rm=TRUE))/s }
smooth_trace <- function(y,wf=0.10,mn=5,mx=21){ y<-as.numeric(y); n<-length(y)
    if(n<3||sum(!is.na(y))<3) return(y); w<-max(mn,min(mx,round(n*wf))); if(w%%2==0) w<-w+1
    h<-(w-1)%/%2; out<-numeric(n); for(i in seq_len(n)){lo<-max(1,i-h);hi<-min(n,i+h);out[i]<-mean(y[lo:hi],na.rm=TRUE)}
    out[is.nan(out)]<-NA; out }

# ─── NA→0 on clinical scores, then sort ───
for (c_ in intersect(c("Braak","CDR"), colnames(prog))) prog[[c_]][is.na(prog[[c_]])] <- 0
plot_df <- prog
plot_df$Study_Group <- factor(plot_df$Study_Group, levels=sg_ordered)
plot_df <- plot_df[order(plot_df$Study_Group, plot_df$Braak, plot_df$CDR), ]
plot_df$x <- seq_len(nrow(plot_df)); rownames(plot_df) <- NULL
cat(sprintf("sorted | %d donors\n", nrow(plot_df)))

# ─── define the 5 traces: column, label, color, heavy? ───
TRACES <- list(
    list(col="mean_Homeo", lab="Homeostatic", color="#17BECF", size=1.1, heavy=FALSE),  # cyan
    list(col="mean_IFN",   lab="IFN (IRM)",   color="#1F77B4", size=1.1, heavy=FALSE),  # blue
    list(col="mean_DAM",   lab="DAM",         color="#D62728", size=1.1, heavy=FALSE),  # red
    list(col="mean_SASP",  lab="SASP",        color="#FF7F0E", size=1.1, heavy=FALSE),  # orange
    list(col="CDR",        lab="CDR (clin.)", color="#2CA02C", size=1.1, heavy=FALSE),  # green
    list(col="pct_sen",    lab="%SnC",        color="#6A0DAD", size=1.8, heavy=TRUE)     # purple (heavy)
)
for (t in TRACES) stopifnot(t$col %in% colnames(plot_df))
for (t in TRACES) stopifnot(t$col %in% colnames(plot_df))

# z-score + smooth each
trace_rows <- list()
for (t in TRACES) {
    z <- z_score(plot_df[[t$col]])
    trace_rows[[t$col]] <- data.frame(
        x=plot_df$x, y=smooth_trace(z), trace=t$lab, color=t$color,
        size=t$size, heavy=t$heavy, stringsAsFactors=FALSE)
}
trace_df <- do.call(rbind, trace_rows); rownames(trace_df) <- NULL
trace_df$trace <- factor(trace_df$trace, levels=sapply(TRACES, `[[`, "lab"))
y_snc <- trace_rows[["pct_sen"]]$y

# ─── bands / labels / dividers / dots ───
band_rows<-list(); label_rows<-list(); divider_x<-numeric(0)
for (sg in sg_ordered){ d<-plot_df[plot_df$Study_Group==sg,]; if(nrow(d)==0) next
    x0<-min(d$x)-0.5; x1<-max(d$x)+0.5; bc<-na_fb(STUDY_GROUP_PALETTE[sg],fallback_color)
    band_rows[[sg]]<-data.frame(xmin=x0,xmax=x1,ymin=-Inf,ymax=Inf,fill=bc,stringsAsFactors=FALSE)
    label_rows[[sg]]<-data.frame(x=(x0+x1)/2,label_main=sg,label_n=sprintf("n=%d",nrow(d)),color=bc,stringsAsFactors=FALSE)}
band_df<-do.call(rbind,band_rows); label_df<-do.call(rbind,label_rows)
for (i in seq_along(sg_ordered)[-length(sg_ordered)]){ d<-plot_df[plot_df$Study_Group==sg_ordered[i],]
    if(nrow(d)>0) divider_x<-c(divider_x,max(d$x)+0.5)}
dot_df<-data.frame(x=plot_df$x, y=y_snc,
    color=na_fb(STUDY_GROUP_PALETTE[as.character(plot_df$Study_Group)],fallback_color))
dot_df<-dot_df[!is.na(dot_df$y),]
y_range<-range(trace_df$y,na.rm=TRUE); y_pad<-0.12*diff(y_range); y_lim<-c(y_range[1]-y_pad,y_range[2]+y_pad)

# ─── main panel ───
p_overall <- ggplot() +
    geom_rect(data=band_df, aes(xmin=xmin,xmax=xmax,ymin=ymin,ymax=ymax,fill=fill), alpha=0.12, inherit.aes=FALSE) +
    scale_fill_identity() +
    {if(length(divider_x)>0) geom_vline(xintercept=divider_x, color="grey60", linetype="dotted", linewidth=0.4)} +
    geom_hline(yintercept=0, color="grey60", linetype="dashed", linewidth=0.4) +
    # non-heavy traces + white halo
    geom_line(data=trace_df[!trace_df$heavy,], aes(x=x,y=y,group=trace), color="white", linewidth=2.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[!trace_df$heavy,], aes(x=x,y=y,color=color,group=trace,linewidth=size)) +
    # %SnC heavy + halo
    geom_line(data=trace_df[trace_df$heavy,], aes(x=x,y=y), color="white", linewidth=3.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[trace_df$heavy,], aes(x=x,y=y,color=color), linewidth=1.8, lineend="round") +
    geom_point(data=dot_df, aes(x=x,y=y,fill=color), shape=21, color="black", size=1.7, stroke=0.3) +
    scale_color_identity()+scale_linewidth_identity() +
    scale_x_continuous(expand=c(0,0)) + scale_y_continuous(limits=y_lim) +
    labs(title=sprintf("Progression: %s | %s | group → Braak → CDR | n=%d", CELL_TYPE, DATASET, nrow(plot_df)),
         x="Donors (group → Braak → CDR)", y="z-score") +
    theme_minimal(base_size=10) +
    theme(plot.title=element_text(size=11,face="bold",color="#333333",hjust=0,margin=margin(b=6)),
          plot.title.position="plot", plot.background=element_rect(fill="white",color=NA),
          panel.border=element_rect(color="#333333",fill=NA,linewidth=0.6),
          panel.grid.major.y=element_line(color="grey92",linewidth=0.3), panel.grid.minor=element_blank(),
          panel.grid.major.x=element_blank(), axis.text.x=element_blank(), axis.ticks.x=element_blank(),
          axis.title.x=element_text(size=8.5,color="#666666",margin=margin(t=6)),
          axis.text.y=element_text(size=9,color="#333333"), axis.title.y=element_text(size=9.5,color="#333333"),
          legend.position="none", plot.margin=margin(8,8,4,8))

# ─── strip ───
strip_plot <- ggplot(label_df) +
    geom_text(aes(x=x,y=1,label=label_main,color=color), size=2.7, fontface="bold", vjust=1) +
    geom_text(aes(x=x,y=0.45,label=label_n,color=color), size=2.4, vjust=1) +
    scale_color_identity() + scale_x_continuous(limits=c(0.5,nrow(plot_df)+0.5),expand=c(0,0)) +
    scale_y_continuous(limits=c(0,1.1)) + theme_void() +
    theme(plot.margin=margin(2,8,0,8), plot.background=element_rect(fill="white",color=NA))

# ─── legend (5 items) ───
leg_df <- data.frame(label=sapply(TRACES,`[[`,"lab"), color=sapply(TRACES,`[[`,"color"),
                     bold=sapply(TRACES,`[[`,"heavy"), stringsAsFactors=FALSE)
n_items<-nrow(leg_df); leg_df$slot<-seq_len(n_items)
leg_df$x_seg_lo<-leg_df$slot-0.5+0.05; leg_df$x_seg_hi<-leg_df$x_seg_lo+0.30; leg_df$x_text<-leg_df$x_seg_hi+0.05
legend_plot <- ggplot(leg_df) +
    geom_segment(aes(x=x_seg_lo,xend=x_seg_hi,y=1,yend=1,color=color), linewidth=0.9, lineend="round") +
    geom_text(aes(x=x_text,y=1,label=label,color=color,fontface=ifelse(bold,"bold","plain")), size=2.7, hjust=0, vjust=0.5) +
    scale_color_identity() +
    scale_x_continuous(limits=c(0.4,n_items+0.6),expand=c(0,0)) + scale_y_continuous(limits=c(0.5,1.5),expand=c(0,0)) +
    theme_void() + theme(plot.margin=margin(2,8,4,8), plot.background=element_rect(fill="white",color=NA))

# ─── compose + save ───
composed <- strip_plot / p_overall / legend_plot + plot_layout(heights=c(0.10,1,0.08))
save_figure(composed, slug=sprintf("%s_progression_5trace", CELL_TYPE), width=9.5, height=4.6)
options(repr.plot.width=10, repr.plot.height=4.9); print(composed)

# ─── peaks ───
cat("\ntrace peaks:\n")
for (t in TRACES) { sm<-smooth_trace(z_score(plot_df[[t$col]])); pk<-which.max(sm)
    cat(sprintf("  %-12s peak idx=%3d group=%-20s z=%+.2f\n", t$lab, pk, as.character(plot_df$Study_Group[pk]), sm[pk])) }

---
## 12 · Paired progression panels

**Why.** %SnC and CDR anchored, each paired with one axis in turn — so the comparison is against a fixed reference rather than between two moving traces.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PAIRED PROGRESSION PANELS — %SnC + CDR anchored, paired with each axis
#   publication quality · 2×2 panels · two-row legend
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })

z_score <- function(v){ v<-as.numeric(v); s<-sd(v,na.rm=TRUE)
    if(is.na(s)||s==0) return(v-mean(v,na.rm=TRUE)); (v-mean(v,na.rm=TRUE))/s }
smooth_trace <- function(y,wf=0.10,mn=5,mx=21){ y<-as.numeric(y); n<-length(y)
    if(n<3) return(y); w<-max(mn,min(mx,round(n*wf))); if(w%%2==0) w<-w+1; h<-(w-1)%/%2
    out<-sapply(seq_len(n),function(i) mean(y[max(1,i-h):min(n,i+h)],na.rm=TRUE)); out[is.nan(out)]<-NA; out }

# ─── NA→0 on clinical scores, sort group → Braak → CDR ───
for (c_ in intersect(c("Braak","CDR"), colnames(prog))) prog[[c_]][is.na(prog[[c_]])] <- 0
plot_df <- prog
plot_df$Study_Group <- factor(plot_df$Study_Group, levels=sg_ordered)
plot_df <- plot_df[order(plot_df$Study_Group, plot_df$Braak, plot_df$CDR), ]
plot_df$x <- seq_len(nrow(plot_df)); rownames(plot_df) <- NULL
cat(sprintf("sorted | %d donors\n", nrow(plot_df)))

# ─── anchors (drawn in every panel) ───
snc_y <- smooth_trace(z_score(plot_df$pct_sen))
cdr_y <- smooth_trace(z_score(plot_df$CDR))

# ─── variables paired against the anchors ───
PAIRS <- list(
    list(col="mean_Homeo", lab="Homeostatic", color="#17BECF"),
    list(col="mean_IFN",   lab="IFN (IRM)",   color="#1F77B4"),
    list(col="mean_DAM",   lab="DAM",         color="#D62728"),
    list(col="mean_SASP",  lab="SASP",        color="#FF7F0E")
)
for (p in PAIRS) stopifnot(p$col %in% colnames(plot_df))

# ─── shared stage bands ───
band_df <- do.call(rbind, lapply(sg_ordered, function(sg){
    d<-plot_df[plot_df$Study_Group==sg,]; if(nrow(d)==0) return(NULL)
    data.frame(xmin=min(d$x)-0.5, xmax=max(d$x)+0.5,
               fill=STUDY_GROUP_PALETTE[sg] %||% fallback_color) }))

# ─── panel builder ───
make_panel <- function(p, show_y=TRUE, show_x=FALSE) {
    var_y <- smooth_trace(z_score(plot_df[[p$col]]))
    d <- rbind(
        data.frame(x=plot_df$x, y=var_y, trace=p$lab,  color=p$color,   lw=0.9),
        data.frame(x=plot_df$x, y=cdr_y, trace="CDR",  color="#2CA02C", lw=0.9),
        data.frame(x=plot_df$x, y=snc_y, trace="%SnC", color="#6A0DAD", lw=1.5))
    # correlation of %SnC with this variable (annotation)
    rho <- suppressWarnings(cor(z_score(plot_df$pct_sen), z_score(plot_df[[p$col]]),
                                method="spearman", use="complete.obs"))
    ggplot() +
        geom_rect(data=band_df, aes(xmin=xmin,xmax=xmax,ymin=-Inf,ymax=Inf,fill=fill),
                  alpha=0.10, inherit.aes=FALSE) + scale_fill_identity() +
        geom_hline(yintercept=0, color="grey70", linetype="dashed", linewidth=0.3) +
        geom_line(data=d, aes(x=x,y=y,group=trace), color="white",
                  linewidth=d$lw+1.0, alpha=0.8, lineend="round") +
        geom_line(data=d, aes(x=x,y=y,color=color,group=trace,linewidth=lw), lineend="round") +
        annotate("text", x=Inf, y=Inf, hjust=1.08, vjust=1.4, size=2.6, color="#555555",
                 label=sprintf("rho[SnC,%s] == %+.2f", gsub("[^A-Za-z]","",p$lab), rho), parse=TRUE) +
        scale_color_identity() + scale_linewidth_identity() +
        scale_x_continuous(expand=c(0,0)) +
        labs(subtitle=p$lab,
             x=if(show_x) "Donors (group \u2192 Braak \u2192 CDR)" else NULL,
             y=if(show_y) "Score (z-score)" else NULL) +
        theme_classic(base_size=10) +
        theme(plot.subtitle=element_text(size=10, face="bold", color="#222222"),
              panel.border=element_rect(color="#333333", fill=NA, linewidth=0.5),
              axis.line=element_blank(),
              axis.text.x=element_blank(), axis.ticks.x=element_blank(),
              axis.text.y=element_text(size=8.5, color="#333333"),
              axis.title=element_text(size=9, color="#333333"),
              plot.margin=margin(6,8,6,8), legend.position="none")
}

# y-label left column only; x-label bottom row only
panels <- list(
    make_panel(PAIRS[[1]], show_y=TRUE,  show_x=FALSE),
    make_panel(PAIRS[[2]], show_y=FALSE, show_x=FALSE),
    make_panel(PAIRS[[3]], show_y=TRUE,  show_x=TRUE),
    make_panel(PAIRS[[4]], show_y=FALSE, show_x=TRUE)
)

# ─── two-row legend, CENTERED ───
leg_df <- data.frame(
    label = c("%SnC (senescent fraction)", "CDR (clinical severity)", "Homeostatic",
              "IFN (IRM)", "DAM", "SASP"),
    color = c("#6A0DAD","#2CA02C","#17BECF","#1F77B4","#D62728","#FF7F0E"),
    stringsAsFactors = FALSE)
leg_df$row <- rep(c(2,1), each=3)[seq_len(nrow(leg_df))]   # row 2 = top
leg_df$col <- rep(1:3, times=2)[seq_len(nrow(leg_df))]

slot_w   <- 3.2
n_cols   <- 3
total_w  <- n_cols * slot_w                 # full laid-out width
x_lim    <- c(0, total_w)                   # plot x-range
# center: the content already spans 0..total_w, so it's centered if x-limits match.
# but the LAST item's text extends past its slot — pad the limits symmetrically
text_pad <- 1.6                             # room for the longest label's text
leg_df$x_seg <- (leg_df$col-1)*slot_w + 0.1
leg_df$x_txt <- leg_df$x_seg + 0.35

legend_plot <- ggplot(leg_df) +
    geom_segment(aes(x=x_seg, xend=x_seg+0.28, y=row, yend=row, color=color),
                 linewidth=1.2, lineend="round") +
    geom_text(aes(x=x_txt, y=row, label=label, color=color),
              size=2.7, hjust=0, vjust=0.5) +
    scale_color_identity() +
    # symmetric x-limits center the content block; pad right for trailing text
    scale_x_continuous(limits=c(-text_pad, total_w + text_pad), expand=c(0,0)) +
    scale_y_continuous(limits=c(0.5, 2.5), expand=c(0,0)) +
    theme_void() + theme(plot.margin=margin(4,8,4,8))
             
# ─── compose + save ───
composed <- (wrap_plots(panels, ncol=2)) / legend_plot +
    plot_layout(heights=c(1, 0.12)) +
    plot_annotation(
        title = "Senescence relative to microglial activation programs across disease progression",
        subtitle = sprintf("%s \u00b7 %s \u00b7 %%SnC and CDR anchored in each panel \u00b7 n=%d donors",
                           CELL_TYPE, DATASET, nrow(plot_df)),
        theme = theme(plot.title=element_text(size=12, face="bold", color="#222222"),
                      plot.subtitle=element_text(size=8.5, color="#666666", margin=margin(b=4))))

save_figure(composed, slug=sprintf("%s_progression_paired", CELL_TYPE), width=10, height=6.5)
options(repr.plot.width=10, repr.plot.height=7); print(composed)

# ─── correlation report ───
cat("\nSpearman rho (%SnC vs each, donor-level):\n")
for (p in PAIRS) {
    rho <- suppressWarnings(cor(plot_df$pct_sen, plot_df[[p$col]], method="spearman", use="complete.obs"))
    cat(sprintf("  %%SnC vs %-12s rho = %+.3f\n", p$lab, rho))
}
rho_cdr <- cor(plot_df$pct_sen, plot_df$CDR, method="spearman", use="complete.obs")
cat(sprintf("  %%SnC vs %-12s rho = %+.3f\n", "CDR", rho_cdr))

In [ ]:
# where does each trace peak along the progression axis? (donor index)
for (v in c("mean_IFN","pct_sen","mean_DAM","mean_SASP")) {
    sm <- smooth_trace(z_score(plot_df[[v]]))
    pk <- which.max(sm)
    cat(sprintf("%-10s peaks at donor idx %3d (%s), Braak=%g\n",
                v, pk, as.character(plot_df$Study_Group[pk]), plot_df$Braak[pk]))
}

---
## 13 · Peak position summary

**Why.** Where each trace peaks along the donor ordering, as a number rather than an eyeball read off the panel.

In [ ]:
suppressPackageStartupMessages({ library(dplyr); library(tidyr); library(ggplot2) })

# ─── config ───
obj        <- mg
STATE_COL  <- "microglia_state"
DONOR_COL  <- "Donor"
GROUP_COL  <- "Study_Group"
GROUP_ORDER <- c("Young_Healthy_Control","Old_Healthy_Control","Old_AD")
GROUP_LAB   <- c(Young_Healthy_Control="Young HC", Old_Healthy_Control="Old HC", Old_AD="Old AD")

md <- obj@meta.data
md[[DONOR_COL]] <- as.character(md[[DONOR_COL]]); md[[STATE_COL]] <- as.character(md[[STATE_COL]])

# per-donor composition, complete 0s (notebook cell 48 logic)
comp <- md %>%
    count(Donor=.data[[DONOR_COL]], Group=.data[[GROUP_COL]], State=.data[[STATE_COL]], name="n") %>%
    group_by(Donor) %>% mutate(pct = 100*n/sum(n)) %>% ungroup() %>%
    complete(nesting(Donor, Group), State, fill=list(n=0, pct=0))

state_order <- comp %>% group_by(State) %>% summarise(t=sum(n)) %>% arrange(desc(t)) %>% pull(State)
comp$State <- factor(comp$State, levels=state_order)
comp$Group <- factor(comp$Group, levels=GROUP_ORDER[GROUP_ORDER %in% unique(comp$Group)])

# state palette: reuse session's state_colors if present, else categorical default
if (exists("state_colors")) {
    state_pal <- state_colors
} else {
    pal_base <- c("#4C78A8","#F58518","#E45756","#72B7B2","#54A24B","#EECA3B","#B279A2","#FF9DA6","#9D755D")
    state_pal <- setNames(pal_base[seq_along(state_order)], state_order)
}

# (A) group-level mean composition (means of within-donor % sum to 100 by construction)
grp_comp <- comp %>% group_by(Group, State) %>% summarise(mean_pct=mean(pct), .groups="drop")
cat("Group-level mean composition (%):\n")
print(grp_comp %>% pivot_wider(names_from=Group, values_from=mean_pct) %>%
        mutate(across(where(is.numeric), ~round(.x,1))) %>% as.data.frame())

p_grp <- ggplot(grp_comp, aes(Group, mean_pct, fill=State)) +
    geom_col(width=0.72, color="white", linewidth=0.3) +
    scale_fill_manual(values=state_pal) + scale_x_discrete(labels=GROUP_LAB) +
    scale_y_continuous(limits=c(0,100), expand=expansion(mult=c(0,0.02))) +
    labs(x=NULL, y="Mean % of microglia", fill="State", title="Microglial state composition by group") +
    theme_classic(base_size=10) +
    theme(panel.border=element_rect(color="#333333",fill=NA,linewidth=0.5), axis.line=element_blank(),
          legend.key.size=unit(0.4,"cm"))

# (B) per-donor stacked bars, split by group (ordered within group by dominant state)
donor_ord <- comp %>% filter(State==state_order[1]) %>% arrange(Group, desc(pct)) %>% pull(Donor)
comp$Donor <- factor(comp$Donor, levels=unique(donor_ord))
p_donor <- ggplot(comp, aes(Donor, pct, fill=State)) +
    geom_col(width=1, color=NA) +
    facet_grid(~Group, scales="free_x", space="free_x", labeller=as_labeller(GROUP_LAB)) +
    scale_fill_manual(values=state_pal) + scale_y_continuous(limits=c(0,100), expand=c(0,0)) +
    labs(x="Donor", y="% of microglia", fill="State", title="Per-donor microglial state composition") +
    theme_classic(base_size=10) +
    theme(axis.text.x=element_blank(), axis.ticks.x=element_blank(),
          strip.background=element_blank(), strip.text=element_text(face="bold",size=9),
          panel.spacing=unit(0.15,"cm"), legend.key.size=unit(0.4,"cm"))

ggsave("mg_state_composition_group.svg", p_grp, width=5, height=4.5, dpi=200, bg="white")
ggsave("mg_state_composition_donor.svg", p_donor, width=9, height=4, dpi=200, bg="white")
options(repr.plot.width=5, repr.plot.height=4.5); print(p_grp)
options(repr.plot.width=9, repr.plot.height=4); print(p_donor)

In [ ]:
suppressPackageStartupMessages({ library(dplyr); library(ggplot2) })

comp_snc$lbl <- ifelse(comp_snc$pct >= 4, sprintf("%.0f%%", comp_snc$pct), "")

p_snc <- ggplot(comp_snc, aes(x=pct, y=Group, fill=State)) +
    geom_col(aes(alpha=SnC, linetype=SnC), width=0.72, color="grey30", linewidth=0.4) +
    geom_text(aes(label=lbl, color=SnC), position=position_stack(vjust=0.5),
              size=2.5, show.legend=FALSE) +
    facet_wrap(~SnC, nrow=1,
               labeller=as_labeller(c(nSnC="Non-senescent", SnC="Senescent"))) +
    scale_fill_manual(values=state_pal) +
    scale_alpha_manual(values=c(nSnC=1, SnC=0.55), guide="none") +
    scale_linetype_manual(values=c(nSnC="solid", SnC="22"), guide="none") +
    scale_color_manual(values=c(nSnC="white", SnC="grey20"), guide="none") +
    scale_y_discrete(limits=rev(levels(comp_snc$Group)), labels=GROUP_LAB) +
    scale_x_continuous(limits=c(0,100), expand=expansion(mult=c(0,0.02))) +
    labs(x="% of microglia", y=NULL, fill="State",
         title="Microglial state composition: non-senescent vs senescent",
         subtitle="State distribution within each group \u00d7 senescence stratum") +
    theme_classic(base_size=10) +
    theme(plot.subtitle=element_text(size=8, color="#666666"),
          strip.background=element_blank(), strip.text=element_text(face="bold", size=10),
          panel.border=element_rect(color="#333333", fill=NA, linewidth=0.5), axis.line=element_blank(),
          legend.key.size=unit(0.4,"cm"))
ggsave("mg_state_composition_by_snc.svg", p_snc, width=9, height=3.6, dpi=200, bg="white")
options(repr.plot.width=9, repr.plot.height=3.6); print(p_snc)